<a href="https://colab.research.google.com/github/wangruig-lang/MLPS_Final_Project/blob/main/model_deep_lstm_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Track A: Deep LSTM + GNN 空间模块
# 特征工程 v2 — Tier A (~28维) / Tier B (~42维)
#
# 运行环境: Colab (T4 GPU) 或本地 (MPS/CPU)
# 预计运行时间: ~15 min (Phase 1 LSTM) + ~10 min (Phase 2 GNN)

---
## §0 Config

In [ ]:
# ============================================================
# §0 Config — 只改这里! 其余 cell 全自动
# ============================================================

MODEL_NAME = "DeepLSTM"
EXPERIMENT_TAG = "v9_rate_w3"        # v8 w=20 过猛 (pred 4万), 降到 w=3 + 抬 threshold

MODEL_TYPE = "lstm"                  # ← 回到 DeepLSTM
GNN_ENABLED = False                  # Phase 1 先不用, Phase 2 (§P2) 自动开启
TWO_STAGE_ENABLED = False
ENSEMBLE_ENABLED = False

# 数据
DATA_DIR = "data"
RESULTS_DIR = "results"
RANDOM_SEED = 42
VALIDATION_SPLIT = 0.2
OUTAGE_LAGS = (1, 3, 6, 12, 24)
ROLLING_WINDOWS = (6, 12, 24)

# 序列
SEQ_LEN = 48
HORIZON = 48
TRAIN_STRIDE = 12                    # ← 保留: 减少滑动窗口冗余

# 训练
BATCH_SIZE = 128
EPOCHS = 120
LEARNING_RATE = 5e-4                 # ← 恢复 v2
WEIGHT_DECAY = 1e-3                  # ← 恢复 v2
LR_SCHEDULER = "plateau"            # ← v4f: val不降时砍LR, 比cosine更快响应
EARLY_STOPPING_PATIENCE = 20        # ← 恢复 v2 (10 太紧)
GRADIENT_CLIP = 1.0
INPUT_NOISE_STD = 0.05               # ← 轻度噪声
MIXUP_ALPHA = 0.2                    # ← 轻度mixup
FEAT_MASK_RATIO = 0.0                # ← 对比组无feat mask

# ---- Deep LSTM 架构 ----
HIDDEN_DIM = 128                     # ← 恢复 v2
NUM_LAYERS = 2                       # ← 恢复 v2
DROPOUT = 0.4                        # ← 恢复 v2
USE_GRU = False
BIDIRECTIONAL = False

# ---- GNN (Phase 2 自动启用) ----
GNN_HIDDEN = 64                      # ← 恢复 v2
GNN_K = 5
SPATIAL_FUSION = "gate"

# Transformer / Seq2Seq (本 notebook 不用, 占位)
N_HEADS = 8
DIM_FEEDFORWARD = 512
USE_CAUSAL_MASK = True
AUTOREGRESSIVE = False
TEACHER_FORCING = 0.5
CLS_THRESHOLD = 0.5
CLS_POS_WEIGHT = 3.0
ENSEMBLE_METHOD = "stacking"

# ---- 特征层级 (v2) ----
FEATURE_TIER = "B"                   # ← 恢复 v2: Tier B ~42 维 (~99%)

# 目标变换
TARGET_TYPE = "rate"                 # "raw" / "log1p" / "rate"  ← rate: 按 tracked 归一化
LOG_TRANSFORM_Y = False              # rate mode 下关掉 log1p

# ---- 损失函数 (v9: 温和版 weighted_mse) ----
# v7 huber → 躺平预测 0 (pred max=1425)
# v8 w=20, th=0.005 → 过度矫正 (pred max=39811, 真值 11903)
# v9 w=3,  th=0.01  → 只加权真正的风暴时段, 加权强度温和
LOSS_FN = "weighted_mse"
HUBER_DELTA = 0.01                   # (huber 路径保留, 不走)
EXTREME_WEIGHT = 3.0                 # ← v9: 20 → 3 (避免过度矫正)
EXTREME_THRESHOLD = 0.01             # ← v9: 0.005 → 0.01 (1% 以上才算风暴)

# 设备
import torch
if torch.cuda.is_available():
    DEVICE = "cuda"
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"=== {MODEL_NAME} ({EXPERIMENT_TAG}) ===")
print(f"Device: {DEVICE} | LSTM: {NUM_LAYERS}层×{HIDDEN_DIM}dim | Dropout: {DROPOUT}")
print(f"Feature Tier: {FEATURE_TIER} | SEQ_LEN: {SEQ_LEN} | HORIZON: {HORIZON}")
print(f"LR: {LEARNING_RATE} | WD: {WEIGHT_DECAY} | Batch: {BATCH_SIZE}")
print(f"TRAIN_STRIDE: {TRAIN_STRIDE} | INPUT_NOISE: {INPUT_NOISE_STD}")
print(f"Loss: {LOSS_FN} | Threshold: {EXTREME_THRESHOLD} | Weight: {EXTREME_WEIGHT}")
print(f"Epochs: {EPOCHS} | Early Stop: {EARLY_STOPPING_PATIENCE}")


=== DeepLSTM (v9_rate_w3) ===
Device: cuda | LSTM: 2层×128dim | Dropout: 0.4
Feature Tier: B | SEQ_LEN: 48 | HORIZON: 48
LR: 0.0005 | WD: 0.001 | Batch: 128
TRAIN_STRIDE: 12 | INPUT_NOISE: 0.05
Loss: weighted_mse | Threshold: 0.01 | Weight: 3.0
Epochs: 120 | Early Stop: 20


---
## §1 数据加载 + 特征工程 (自动，不用改)

In [ ]:
# §1.1 导入 + 环境配置

# ---- Colab 自动检测 ----
import os, sys
IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    # Colab: 安装依赖 + 挂载 Drive
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'netCDF4', 'xarray', 'tqdm'], check=True)

    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # 数据路径: 把 train.nc 放在 Google Drive 的 MLPS_Final_Project/data/ 下
    DRIVE_PROJECT = '/content/drive/MyDrive/MLPS_Final_Project'
    if os.path.exists(os.path.join(DRIVE_PROJECT, 'data', 'train.nc')):
        DATA_DIR = os.path.join(DRIVE_PROJECT, 'data')
        RESULTS_DIR = os.path.join(DRIVE_PROJECT, 'results')
        print(f"✓ 使用 Drive 数据: {DATA_DIR}")
    elif os.path.exists('data/train.nc'):
        print("✓ 使用本地 data/")
    else:
        print("⚠ 找不到 train.nc! 请上传到 data/ 或 Drive/MLPS_Final_Project/data/")

    print(f"✓ Colab 环境就绪")
else:
    print(f"✓ 本地环境")

import time, warnings, platform
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from datetime import datetime
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if DEVICE == 'cuda': torch.cuda.manual_seed_all(RANDOM_SEED)

# 中文字体
if platform.system() == 'Darwin':
    plt.rcParams['font.sans-serif'] = ['PingFang SC', 'Heiti TC', 'Arial Unicode MS']
elif platform.system() == 'Linux':
    plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei', 'Noto Sans CJK SC']
plt.rcParams['axes.unicode_minus'] = False

print(f"PyTorch {torch.__version__} | Device: {DEVICE}")
if DEVICE == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    mem = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f"  Memory: {mem / 1e9:.1f} GB")

Mounted at /content/drive
✓ 使用 Drive 数据: /content/drive/MyDrive/MLPS_Final_Project/data
✓ Colab 环境就绪
PyTorch 2.10.0+cu128 | Device: cuda
  GPU: Tesla T4
  Memory: 15.6 GB


In [ ]:
# §0 Skip switch — HistGBM-only mode
# 跑 HistGBM 链路时跳过所有 LSTM/GNN 训练相关 cell, 节省 Colab 时间
SKIP_LSTM_TRAIN = True   # 跳过 §2.2/§3/§4/§5/§6/§7.2/§P2
SKIP_PART2_GNN  = True   # 跳过 §P2.1-§P2.5 (Part II GNN)
print(f'[CONFIG] SKIP_LSTM_TRAIN={SKIP_LSTM_TRAIN}  SKIP_PART2_GNN={SKIP_PART2_GNN}')
print('  → 跑 §1.1→§1.5→§2.1→§7.1→§7.3(stub LSTM)→§7.4b→§7.5→§7.6 就够了')


[CONFIG] SKIP_LSTM_TRAIN=True  SKIP_PART2_GNN=True
  → 跑 §1.1→§1.5→§2.1→§7.1→§7.3(stub LSTM)→§7.4b→§7.5→§7.6 就够了


In [ ]:
# §1.2 加载数据
ds = xr.open_dataset(os.path.join(DATA_DIR, 'train.nc'))
timestamps = pd.to_datetime(ds.timestamp.values)
locations = list(ds.location.values)
weather_features = list(ds.feature.values)
outage  = ds.out.transpose('timestamp','location').values.astype(float)
tracked = ds.tracked.transpose('timestamp','location').values.astype(float)
weather = ds.weather.transpose('timestamp','location','feature').values.astype(float)
T, L = outage.shape
print(f"T={T}h | L={L} counties | F={len(weather_features)} weather features")
print(f"时间范围: {timestamps[0]} ~ {timestamps[-1]}")

T=2161h | L=83 counties | F=109 weather features
时间范围: 2023-04-01 00:00:00 ~ 2023-06-30 00:00:00


In [ ]:
# §1.3 Phase 1 关键产出 (GMM 分层 + 天气特征筛选)
from sklearn.mixture import GaussianMixture
from scipy.stats import spearmanr

# GMM regime thresholds
nonzero_out = outage[outage > 0].flatten()
log_nz = np.log1p(nonzero_out).reshape(-1, 1)
best_k, best_bic = 2, np.inf
for k in range(2, 6):
    gm = GaussianMixture(n_components=k, random_state=RANDOM_SEED, max_iter=300).fit(log_nz)
    bic = gm.bic(log_nz)
    if bic < best_bic: best_k, best_bic = k, bic
gm_final = GaussianMixture(n_components=best_k, random_state=RANDOM_SEED, max_iter=300).fit(log_nz)
sorted_means = np.sort(gm_final.means_.flatten())
gmm_thresholds = [int(np.expm1((sorted_means[i]+sorted_means[i+1])/2)) for i in range(len(sorted_means)-1)]
print(f"GMM k={best_k}, thresholds={gmm_thresholds}")

# Weather feature selection (Spearman + collinearity dedup)
flat_out = outage.flatten()
sample_n = min(200_000, len(flat_out))
sidx = np.random.choice(len(flat_out), sample_n, replace=False)
corr_dict = {}
for fi, fn in enumerate(weather_features):
    rho, _ = spearmanr(flat_out[sidx], weather[:,:,fi].flatten()[sidx])
    corr_dict[fn] = abs(rho)
corr_series = pd.Series(corr_dict).sort_values(ascending=False)

selected_features = []
for feat in corr_series.index:
    fi = weather_features.index(feat)
    if not any(abs(np.corrcoef(
        weather[:,:,fi].flatten()[sidx],
        weather[:,:,weather_features.index(s)].flatten()[sidx])[0,1]) > 0.85
        for s in selected_features):
        selected_features.append(feat)
top_weather_features = list(corr_series.head(6).index)
print(f"去冗余后: {len(selected_features)} features | Top-6: {top_weather_features}")

GMM k=5, thresholds=[2, 14, 76, 403]
去冗余后: 88 features | Top-6: ['lai', 'cape_1', 'blh', 'orog', 't', 'sdlwrf']


In [ ]:

import time
import numpy as np
import pandas as pd

def build_features(ds, sel_weather, top_weather, thresholds,
                   o_lags=(1, 3, 6, 12, 24), r_wins=(6, 12, 24),
                   weather_lags=(1, 3, 6, 12, 24), max_weather_feats=8):
    """v3 特征构造: 去掉 PCA，只保留 causal weather raw features。"""
    ts = pd.to_datetime(ds.timestamp.values)
    locs = list(ds.location.values)
    fnames = list(ds.feature.values)
    out = ds.out.transpose('timestamp', 'location').values.astype(np.float32)
    trk = ds.tracked.transpose('timestamp', 'location').values.astype(np.float32)
    w = ds.weather.transpose('timestamp', 'location', 'feature').values.astype(np.float32)
    Tl, Ll = out.shape

    sel_weather = [f for f in (sel_weather or fnames) if f in fnames]
    top_weather = list(top_weather or [])

    # 强制保留一组 storm 相关天气变量，避免 PCA 把事件触发信号压扁。
    forced_weather = ['u10', 'v10', 'gust', 'tp', 'crain', 'csnow', 't2m', 'sp']
    weather_pool = []
    for f in top_weather + forced_weather + sel_weather:
        if f in fnames and f not in weather_pool:
            weather_pool.append(f)
        if len(weather_pool) >= max_weather_feats:
            break
    if not weather_pool:
        weather_pool = fnames[:min(max_weather_feats, len(fnames))]

    precip_like = {'tp', 'cp', 'lsp', 'crain', 'csnow'}
    binary_like = {'crain', 'csnow'}

    print(f'  weather raw features ({len(weather_pool)}): {weather_pool}')
    pca_info = {
        'pca_model': None,
        'w_mean': None,
        'w_std': None,
        'n_components': 0,
        'explained_variance_ratio': np.array([], dtype=np.float32),
        'cumulative_variance': np.array([], dtype=np.float32),
        'note': 'PCA disabled; using causal raw-weather features',
    }

    records = []
    weather_generated_cols = []

    for li, loc in enumerate(locs):
        d = pd.DataFrame({
            'timestamp': ts,
            'location': loc,
            'out': out[:, li],
            'tracked': trk[:, li],
        })
        d['hour_sin'] = np.sin(2 * np.pi * ts.hour / 24)
        d['hour_cos'] = np.cos(2 * np.pi * ts.hour / 24)
        d['dow_sin'] = np.sin(2 * np.pi * ts.dayofweek / 7)
        d['dow_cos'] = np.cos(2 * np.pi * ts.dayofweek / 7)

        o_s = d['out']
        rate = d['out'] / d['tracked'].replace(0, np.nan)

        for lag in o_lags:
            d[f'out_lag_{lag}h'] = o_s.shift(lag)
            d[f'rate_lag_{lag}h'] = rate.shift(lag)

        so = o_s.shift(1)
        for win in r_wins:
            roll = so.rolling(win, min_periods=1)
            d[f'out_rmean_{win}h'] = roll.mean()
            d[f'out_rmax_{win}h'] = roll.max()
            d[f'out_rstd_{win}h'] = roll.std().fillna(0)
            d[f'out_rsum_{win}h'] = roll.sum()

        for bw in [6, 12, 24]:
            d[f'had_outage_{bw}h'] = (so.rolling(bw, min_periods=1).max() > 0).astype(int)

        if thresholds:
            lo = o_s.shift(1)
            reg = pd.Series(np.zeros(Tl, dtype=np.float32), index=d.index)
            for th in thresholds:
                reg = reg + (lo > th).astype(np.float32)
            d['out_regime'] = reg
            d['out_regime_max_24h'] = d['out_regime'].rolling(24, min_periods=1).max()

        local_weather_cols = []
        for fn in weather_pool:
            fi = fnames.index(fn)
            s = pd.Series(w[:, li, fi], index=d.index, dtype=np.float32)
            s1 = s.shift(1)

            base_cols = {
                f'w_{fn}_lag_1h': s.shift(1),
                f'w_{fn}_lag_3h': s.shift(3),
                f'w_{fn}_lag_6h': s.shift(6),
                f'w_{fn}_lag_12h': s.shift(12),
                f'w_{fn}_lag_24h': s.shift(24),
                f'w_{fn}_diff_3h': s1 - s1.shift(3),
                f'w_{fn}_diff_6h': s1 - s1.shift(6),
                f'w_{fn}_rmean_6h': s1.rolling(6, min_periods=1).mean(),
                f'w_{fn}_rmax_6h': s1.rolling(6, min_periods=1).max(),
                f'w_{fn}_rstd_6h': s1.rolling(6, min_periods=1).std().fillna(0),
                f'w_{fn}_rmean_24h': s1.rolling(24, min_periods=1).mean(),
                f'w_{fn}_rmax_24h': s1.rolling(24, min_periods=1).max(),
            }
            for col, values in base_cols.items():
                d[col] = values
                local_weather_cols.append(col)

            if fn in precip_like:
                extra_cols = {
                    f'w_{fn}_sum_6h': s1.rolling(6, min_periods=1).sum(),
                    f'w_{fn}_sum_12h': s1.rolling(12, min_periods=1).sum(),
                    f'w_{fn}_sum_24h': s1.rolling(24, min_periods=1).sum(),
                }
                for col, values in extra_cols.items():
                    d[col] = values
                    local_weather_cols.append(col)

            if fn in binary_like:
                col = f'w_{fn}_any_6h'
                d[col] = (s1.rolling(6, min_periods=1).max() > 0).astype(np.float32)
                local_weather_cols.append(col)

        # 一个简单但 causal 的 storm 强度摘要。
        shock_cols = [c for c in local_weather_cols if c.endswith('_diff_3h')][:4]
        if shock_cols:
            d['weather_shock_score'] = d[shock_cols].abs().mean(axis=1)
            d['weather_shock_max_6h'] = d['weather_shock_score'].rolling(6, min_periods=1).max()
            d['n_weather_jump'] = (d[shock_cols].abs() > 0).sum(axis=1).astype(np.float32)
            local_weather_cols += ['weather_shock_score', 'weather_shock_max_6h', 'n_weather_jump']

        if not weather_generated_cols:
            weather_generated_cols = local_weather_cols.copy()

        records.append(d)

    df = pd.concat(records, ignore_index=True)

    lag_cols = [f'out_lag_{l}h' for l in o_lags] + [f'rate_lag_{l}h' for l in o_lags]
    roll_cols = []
    for win in r_wins:
        roll_cols += [
            f'out_rmean_{win}h',
            f'out_rmax_{win}h',
            f'out_rstd_{win}h',
            f'out_rsum_{win}h',
        ]
    roll_cols += [f'had_outage_{bw}h' for bw in [6, 12, 24]]

    storm_cols = [c for c in ['weather_shock_score', 'weather_shock_max_6h', 'n_weather_jump'] if c in df.columns]
    regime_cols = ['out_regime', 'out_regime_max_24h'] if thresholds else []
    time_cols = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']

    key_weather_cols = []
    for fn in weather_pool[:4]:
        for suffix in ['lag_1h', 'lag_3h', 'diff_3h', 'rmax_6h', 'rmean_24h']:
            c = f'w_{fn}_{suffix}'
            if c in df.columns:
                key_weather_cols.append(c)

    tier_compact = [c for c in lag_cols + roll_cols + storm_cols + key_weather_cols if c in df.columns]
    tier_full = [c for c in tier_compact + regime_cols + time_cols + weather_generated_cols if c in df.columns]

    print(f'  Tier A: {len(tier_compact)} dims | Tier B: {len(tier_full)} dims')
    return df, tier_compact, tier_full, pca_info


print('构造特征中...')
t0 = time.time()
df, tier_compact, tier_full, pca_info = build_features(
    ds, selected_features, top_weather_features, gmm_thresholds,
    OUTAGE_LAGS, ROLLING_WINDOWS
)
print(f'✓ {time.time()-t0:.0f}s | {df.shape[0]:,} rows')

构造特征中...
  weather raw features (8): ['lai', 'cape_1', 'blh', 'orog', 't', 'sdlwrf', 'u10', 'v10']
  Tier A: 48 dims | Tier B: 153 dims
✓ 6s | 179,363 rows


In [ ]:
# §1.5 特征选择 + 训练/验证划分 + 标准化

# ---- 根据 FEATURE_TIER 选择特征列 (v2) ----
if FEATURE_TIER == "A":
    feature_cols = tier_compact
elif FEATURE_TIER == "B":
    feature_cols = tier_full
else:
    feature_cols = tier_full
    print(f'⚠ 未知 FEATURE_TIER="{FEATURE_TIER}", 使用 Tier B')

print(f'FEATURE_TIER = "{FEATURE_TIER}" → {len(feature_cols)} 维特征')

# ---- 训练/验证划分 (交替日 split, 修复分布不匹配) ----
# 原来"前80%训练+后20%验证"会把6月下旬的大风暴全部放到val, train完全没见过
# 改为: 每5天取1天做val, train/val都覆盖完整时间范围
SPLIT_MODE = "interleaved"  # "interleaved" 或 "chronological"

if SPLIT_MODE == "interleaved":
    days_since_start = (df['timestamp'] - df['timestamp'].min()).dt.days
    # 每 1/VALIDATION_SPLIT 天里, 第 1 天做 val (例如 20% → 每5天第1天)
    stride = int(round(1.0 / VALIDATION_SPLIT))
    is_val = (days_since_start % stride == 0)
    train_df = df[~is_val].dropna(subset=feature_cols).copy()
    val_df = df[is_val].dropna(subset=feature_cols).copy()
    print(f'SPLIT_MODE = interleaved (每{stride}天取第1天做val)')
else:
    split_idx = int(T * (1 - VALIDATION_SPLIT))
    split_time = timestamps[split_idx]
    train_df = df[df['timestamp'] < split_time].dropna(subset=feature_cols).copy()
    val_df = df[df['timestamp'] >= split_time].dropna(subset=feature_cols).copy()
    print(f'SPLIT_MODE = chronological')

scaler = StandardScaler().fit(train_df[feature_cols].values)
print(f'Train: {len(train_df):,} | Val: {len(val_df):,}')
print(f'y_train: mean={train_df["out"].mean():.1f}, nonzero={train_df["out"].gt(0).mean():.1%}')
print(f'y_val:   mean={val_df["out"].mean():.1f}, nonzero={val_df["out"].gt(0).mean():.1%}')

FEATURE_TIER = "B" → 153 维特征
SPLIT_MODE = interleaved (每5天取第1天做val)
Train: 141,148 | Val: 33,034
y_train: mean=46.7, nonzero=29.5%
y_val:   mean=32.9, nonzero=30.0%


---
## §2 DataLoader

In [ ]:
# §2.1 DataFrame → 3D 张量 (支持 log1p / rate target)
def df_to_3d(sub_df, locs, feat_cols, scaler_obj, target_mode='raw'):
    """
    target_mode:
      'raw'   - y = out (原始停电数)
      'log1p' - y = log1p(out)
      'rate'  - y = out / tracked  (停电率, 0-1)
    """
    ts_list = sorted(sub_df['timestamp'].unique())
    ts2i = {ts:i for i,ts in enumerate(ts_list)}
    loc2i = {l:i for i,l in enumerate(locs)}
    Ts, Ls, Fs = len(ts_list), len(locs), len(feat_cols)
    X = np.zeros((Ts,Ls,Fs), dtype=np.float32)
    Y = np.zeros((Ts,Ls), dtype=np.float32)
    TRK = np.zeros((Ts,Ls), dtype=np.float32)  # 保存 tracked 用于 rate 还原
    sorted_df = sub_df.sort_values(['timestamp','location'])
    scaled = scaler_obj.transform(sorted_df[feat_cols].fillna(0).values)
    for idx, (_, row) in enumerate(sorted_df.iterrows()):
        ti, li = ts2i.get(row['timestamp']), loc2i.get(row['location'])
        if ti is not None and li is not None:
            X[ti,li,:] = scaled[idx]
            trk = row['tracked'] if row['tracked'] > 0 else 1.0
            TRK[ti,li] = trk
            if target_mode == 'log1p':
                Y[ti,li] = np.log1p(row['out'])
            elif target_mode == 'rate':
                Y[ti,li] = row['out'] / trk
            else:
                Y[ti,li] = row['out']
    return X, Y, TRK

# 根据 config 决定 target mode
TARGET_MODE = 'rate' if TARGET_TYPE == 'rate' else ('log1p' if LOG_TRANSFORM_Y else 'raw')
print(f'TARGET_MODE = {TARGET_MODE}')

X_train_3d, Y_train_3d, TRK_train_3d = df_to_3d(train_df, locations, feature_cols, scaler, TARGET_MODE)
X_val_3d,   Y_val_3d,   TRK_val_3d   = df_to_3d(val_df,   locations, feature_cols, scaler, TARGET_MODE)

# 保留原始空间 Y_val (评估用)
_, Y_val_3d_raw, _ = df_to_3d(val_df, locations, feature_cols, scaler, 'raw')

print(f'X_train: {X_train_3d.shape} | X_val: {X_val_3d.shape}')
print(f'  Y range: [{Y_train_3d.min():.3f}, {Y_train_3d.max():.3f}] (mode={TARGET_MODE})')
if TARGET_MODE == 'rate':
    print(f'  TRK range: [{TRK_val_3d.min():.0f}, {TRK_val_3d.max():.0f}]')

TARGET_MODE = rate
X_train: (1728, 83, 153) | X_val: (409, 83, 153)
  Y range: [0.000, 0.543] (mode=rate)
  TRK range: [0, 922425]


In [ ]:
if SKIP_LSTM_TRAIN:
    print('[SKIPPED] §2.2 DataLoader (SKIP_LSTM_TRAIN=True)')
else:
    # §2.2 PyTorch Dataset (支持 stride + 输入噪声 + 特征屏蔽)
    class OutageDataset(Dataset):
        """
        mode='per_county':    每样本 = 单县时间窗 → (seq_len, F) → (horizon,)
        mode='all_counties':  每样本 = 全部县同一时间窗 → (seq_len, L, F) → (horizon, L)

        stride: 训练集用 stride>1 去掉冗余 (相邻样本 overlap 太多导致过拟合)
        noise_std: 训练时给输入加高斯噪声 (数据增强)
        feat_mask_ratio: 随机屏蔽特征列比例 (防止过度依赖单一特征)
        """
        def __init__(self, X, Y, seq_len, horizon, mode='per_county', stride=1,
                     noise_std=0.0, feat_mask_ratio=0.0):
            self.X, self.Y = X, Y
            self.seq_len, self.horizon, self.mode = seq_len, horizon, mode
            self.noise_std = noise_std
            self.feat_mask_ratio = feat_mask_ratio
            self.Td, self.Ld, self.Fd = X.shape
            self.starts = list(range(0, self.Td - seq_len - horizon + 1, stride))
            self.n = len(self.starts) * (self.Ld if mode == 'per_county' else 1)

        def __len__(self): return self.n

        def __getitem__(self, idx):
            if self.mode == 'per_county':
                si = idx // self.Ld  # which start index
                li = idx % self.Ld   # which county
                t = self.starts[si]
                x = self.X[t:t+self.seq_len, li, :]                     # (S, F)
                y = self.Y[t+self.seq_len:t+self.seq_len+self.horizon, li]  # (H,)
            else:
                t = self.starts[idx]
                x = self.X[t:t+self.seq_len]                            # (S, L, F)
                y = self.Y[t+self.seq_len:t+self.seq_len+self.horizon]  # (H, L)

            x = torch.tensor(x, dtype=torch.float32)
            y = torch.tensor(y, dtype=torch.float32)

            # 训练时加噪声 (数据增强, 防过拟合)
            if self.noise_std > 0:
                x = x + torch.randn_like(x) * self.noise_std

            # 训练时随机屏蔽特征列 (类似 feature dropout)
            if self.feat_mask_ratio > 0:
                n_feat = x.shape[-1]
                n_mask = max(1, int(n_feat * self.feat_mask_ratio))
                mask_idx = torch.randperm(n_feat)[:n_mask]
                x[..., mask_idx] = 0.0

            return x, y

    # GNN 需要 all_counties，其余用 per_county
    DS_MODE = 'all_counties' if GNN_ENABLED else 'per_county'

    train_ds = OutageDataset(X_train_3d, Y_train_3d, SEQ_LEN, HORIZON,
                             mode=DS_MODE, stride=TRAIN_STRIDE,
                             noise_std=INPUT_NOISE_STD, feat_mask_ratio=FEAT_MASK_RATIO)
    val_ds   = OutageDataset(X_val_3d,   Y_val_3d,   SEQ_LEN, HORIZON,
                             mode=DS_MODE, stride=1, noise_std=0.0, feat_mask_ratio=0.0)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=(DEVICE=='cuda'))
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE*2, shuffle=False, num_workers=0)

    xb, yb = next(iter(train_loader))
    print(f'Mode: {DS_MODE} | Stride: {TRAIN_STRIDE} | Noise: {INPUT_NOISE_STD} | FeatMask: {FEAT_MASK_RATIO}')
    print(f'Mixup alpha: {MIXUP_ALPHA}')
    print(f'Train: {len(train_ds):,} (stride={TRAIN_STRIDE}) | Val: {len(val_ds):,} (stride=1)')
    print(f'Batch X: {xb.shape} | Batch Y: {yb.shape}')
    print(f'  (去冗余前 train 约 {len(val_ds) * (1-VALIDATION_SPLIT)/VALIDATION_SPLIT:,.0f} 样本, '
          f'现在 {len(train_ds):,}, 减少 {1 - len(train_ds)/(len(val_ds)*(1-VALIDATION_SPLIT)/VALIDATION_SPLIT):.0%})')

[SKIPPED] §2.2 DataLoader (SKIP_LSTM_TRAIN=True)


---
## §3 模型定义

### 关于 BiLSTM: 用不用?

**结论: 本任务不建议用 BiLSTM，用单向 LSTM 即可。**

原因: BiLSTM 的反向 LSTM 需要看到**未来**的数据来产生 hidden state。
在时序预测中，推理时我们只有过去的数据，没有未来 —— 所以 BiLSTM 的反向部分
在训练时能利用未来信息 (看起来效果好)，但在推理时退化成用全零 hidden state，
等于白训了一半的参数。

> **什么时候该用 BiLSTM?** NLP (整个句子都已知)、分类任务 (输入是完整序列)。
> **时序预测?** 单向 LSTM 就够了。把省下来的参数加到深度或宽度上更划算。

---

### GNN 讲解 (给只学过 CNN 和 RNN 的同学)

**一句话**: CNN 处理网格数据 (图片), RNN 处理序列数据 (时间), GNN 处理**图结构**数据 (节点+边)。

#### 你已经懂的东西

| | CNN | RNN | GNN |
|---|---|---|---|
| **数据结构** | 网格 (像素) | 序列 (时间步) | 图 (节点+边) |
| **核心操作** | 卷积核滑窗 | hidden state 传递 | 邻居信息聚合 |
| **学到什么** | 局部空间模式 | 时间依赖 | **节点间的关系** |

#### 为什么停电预测需要 GNN?

83 个县不是独立的! 一场风暴从西向东扫过 Michigan，Wayne County 先停电，
隔壁 Oakland County 2 小时后也停了。纯 LSTM 把每个县独立预测，看不到这种
**空间传播**。

GNN 做的事: 让每个县"看看邻居的情况"，然后更新自己的预测。

#### GCN 一层的数学 (和 CNN 类比)

```
CNN:  output[i,j] = Σ kernel[m,n] × input[i+m, j+n]     # 对邻居像素加权求和
GCN:  output[county_i] = Σ A[i,j] × W × input[county_j]  # 对邻居县加权求和
```

- `A[i,j]` = 邻接矩阵，定义哪些县是"邻居" (类似 CNN 的卷积核覆盖范围)
- `W` = 可学习权重矩阵 (类似 CNN 的卷积核参数)
- 叠 2 层 GCN = 每个县能看到 2-hop 范围内的所有县 (类似 CNN 增大感受野)

#### 本 notebook 的架构

```
输入: (B, T, 83, F) — B个样本, T个时间步, 83个县, F个特征

Step 1: LSTM (时间维度)
  对每个县独立跑 LSTM → 得到每个县的时间编码 h_lstm[i]
  
Step 2: GCN (空间维度)  
  h_gnn[i] = ReLU(Σ_j A[i,j] × W × h_lstm[j])
  # 每个县聚合邻居的 LSTM 编码
  
Step 3: 融合
  h_final[i] = Gate(h_lstm[i], h_gnn[i])
  # 用门控决定时间信息和空间信息的比例
  
Step 4: 预测
  pred[i] = Linear(h_final[i])
```

In [ ]:
if SKIP_LSTM_TRAIN:
    print('[SKIPPED] §3.1 baseline models (SKIP_LSTM_TRAIN=True)')
else:
    # §3.1 三个基础模型 + SimpleSeq2Seq (对比组 baseline)

    class SimpleSeq2Seq(nn.Module):
        """
        对比组的简单 baseline: LSTM + Linear head
        无 dropout / 无 residual / 无 LayerNorm
        """
        def __init__(self, input_dim, hidden_dim=64, num_layers=1, horizon=24, **kwargs):
            super().__init__()
            self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
            self.head = nn.Linear(hidden_dim, horizon)
        def forward(self, x):
            _, (h, _) = self.lstm(x)
            return self.head(h[-1])


    class DeepLSTM(nn.Module):
        """
        加强版多层 LSTM
        - LayerNorm 稳定深层训练
        - Residual connection (层间跳跃连接)
        - 简洁 MLP head (防过拟合)
        """
        def __init__(self, input_dim, hidden_dim, num_layers, horizon,
                     dropout=0.2, use_gru=False, bidirectional=False):
            super().__init__()
            ndir = 2 if bidirectional else 1
            self.hidden_dim = hidden_dim
            self.ndir = ndir

            RNN = nn.GRU if use_gru else nn.LSTM
            self.rnn = RNN(input_dim, hidden_dim, num_layers, batch_first=True,
                           dropout=dropout if num_layers > 1 else 0, bidirectional=bidirectional)

            self.residual_proj = nn.Linear(input_dim, hidden_dim * ndir)
            self.norm = nn.LayerNorm(hidden_dim * ndir)
            self.drop = nn.Dropout(dropout)

            # 简化 head: 1 层隐藏 + 输出 (之前 3 层太重, 过拟合)
            hdim = hidden_dim * ndir
            self.head = nn.Sequential(
                nn.Linear(hdim, hdim // 2),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hdim // 2, horizon),
            )

        def forward(self, x):
            """x: (B, T, F) → (B, horizon)"""
            rnn_out, _ = self.rnn(x)              # (B, T, H*ndir)
            last = rnn_out[:, -1, :]              # (B, H*ndir)
            res = self.residual_proj(x[:, -1, :]) # (B, H*ndir)
            last = self.drop(self.norm(last + res))
            return self.head(last)


    class Seq2SeqAttention(nn.Module):
        def __init__(self, input_dim, hidden_dim, num_layers, horizon, dropout=0.2):
            super().__init__()
            self.encoder = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True,
                                   dropout=dropout if num_layers > 1 else 0)
            self.attn_We = nn.Linear(hidden_dim, hidden_dim, bias=False)
            self.attn_Wd = nn.Linear(hidden_dim, hidden_dim, bias=False)
            self.attn_v  = nn.Linear(hidden_dim, 1, bias=False)
            self.head = nn.Sequential(
                nn.Linear(hidden_dim*2, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
                nn.Linear(hidden_dim, horizon))
        def attention(self, enc_out, h):
            e = torch.tanh(self.attn_We(enc_out) + self.attn_Wd(h).unsqueeze(1))
            w = torch.softmax(self.attn_v(e).squeeze(-1), dim=1)
            return torch.bmm(w.unsqueeze(1), enc_out).squeeze(1), w
        def forward(self, x, return_attention=False):
            enc_out, (h_n, _) = self.encoder(x)
            ctx, aw = self.attention(enc_out, h_n[-1])
            out = self.head(torch.cat([h_n[-1], ctx], dim=1))
            return (out, aw) if return_attention else out

    class TransformerForecaster(nn.Module):
        def __init__(self, input_dim, d_model, n_heads, num_layers, horizon,
                     dim_ff=512, dropout=0.1, max_len=200, causal=True):
            super().__init__()
            self.causal = causal
            self.proj = nn.Linear(input_dim, d_model)
            pe = torch.zeros(max_len, d_model)
            pos = torch.arange(max_len).unsqueeze(1).float()
            div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0)/d_model))
            pe[:,0::2] = torch.sin(pos*div)
            pe[:,1::2] = torch.cos(pos*div[:d_model//2]) if d_model%2 else torch.cos(pos*div)
            self.register_buffer('pe', pe.unsqueeze(0))
            self.drop = nn.Dropout(dropout)
            el = nn.TransformerEncoderLayer(d_model, n_heads, dim_ff, dropout, batch_first=True, activation='gelu')
            self.encoder = nn.TransformerEncoder(el, num_layers)
            self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, d_model),
                                      nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model, horizon))
        def forward(self, x):
            B,T,_ = x.shape
            x = self.drop(self.proj(x) + self.pe[:,:T])
            mask = torch.triu(torch.ones(T,T,device=x.device),1).bool() if self.causal else None
            return self.head(self.encoder(x, mask=mask)[:,-1,:])

    print('✓ DeepLSTM (v2: 简化 head + dropout) / Seq2Seq / Transformer')

[SKIPPED] §3.1 baseline models (SKIP_LSTM_TRAIN=True)


In [ ]:
if SKIP_LSTM_TRAIN:
    print('[SKIPPED] §3.2 GNN module (SKIP_LSTM_TRAIN=True)')
else:
    # §3.2 GNN 空间模块

    # ======================== GCN 层 ========================
    class GCNLayer(nn.Module):
        """
        一层 Graph Convolutional Network

        类比 CNN:
          CNN:  对邻居像素做加权求和 (卷积核固定 3×3)
          GCN:  对邻居节点做加权求和 (邻接矩阵定义谁是邻居)

        公式: H' = ReLU(A_norm @ H @ W + b)
          A_norm: 归一化邻接矩阵 (83×83), 预计算
          H: 节点特征 (83×in_dim)
          W: 可学习权重 (in_dim×out_dim)
        """
        def __init__(self, in_dim, out_dim):
            super().__init__()
            self.W = nn.Linear(in_dim, out_dim, bias=False)
            self.b = nn.Parameter(torch.zeros(out_dim))

        def forward(self, x, A):
            """
            x: (B, 83, in_dim) — 每个县的特征向量
            A: (83, 83)         — 归一化邻接矩阵
            """
            h = self.W(x)                # (B, 83, out_dim) — 线性变换
            h = torch.matmul(A, h)       # (B, 83, out_dim) — 邻居聚合 (图卷积核心!)
            return torch.relu(h + self.b)


    # ======================== LSTM + GNN 融合模型 ========================
    class SpatialWrapper(nn.Module):
        """
        在任何时序 backbone 上叠加 GNN 空间模块。

        架构:
          (B, T, L, F)
            ↓ 每县独立 LSTM 编码
          (B, L, H_lstm)
            ↓ 2 层 GCN 聚合邻居信息
          (B, L, H_gnn)
            ↓ 门控融合 LSTM + GCN
          (B, L, H_fused)
            ↓ MLP 预测头
          (B, L, horizon)
        """
        def __init__(self, backbone, hidden_dim, horizon, n_counties,
                     gnn_hidden=64, fusion='gate', dropout=0.2):
            super().__init__()
            self.backbone = backbone  # DeepLSTM 的 rnn 部分
            self.n_counties = n_counties
            self.fusion = fusion

            # 从 backbone 获取 rnn 的输出维度
            rnn_out_dim = hidden_dim * (2 if hasattr(backbone, 'ndir') and backbone.ndir == 2 else 1)

            # 2 层 GCN (2-hop 邻居)
            self.gcn1 = GCNLayer(rnn_out_dim, gnn_hidden)
            self.gcn2 = GCNLayer(gnn_hidden, gnn_hidden)

            # 融合方式
            if fusion == 'concat':
                head_in = rnn_out_dim + gnn_hidden
            elif fusion == 'gate':
                # 门控: g = sigmoid(W[h_lstm; h_gnn]) → h = g⊙h_lstm + (1-g)⊙proj(h_gnn)
                self.gate_fc = nn.Linear(rnn_out_dim + gnn_hidden, rnn_out_dim)
                self.gnn_proj = nn.Linear(gnn_hidden, rnn_out_dim)
                head_in = rnn_out_dim
            else:  # add
                self.gnn_proj = nn.Linear(gnn_hidden, rnn_out_dim)
                head_in = rnn_out_dim

            self.head = nn.Sequential(
                nn.LayerNorm(head_in),
                nn.Linear(head_in, hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, hidden_dim // 2),
                nn.GELU(),
                nn.Linear(hidden_dim // 2, horizon),
            )

        def forward(self, x, adj):
            """
            x: (B, T, L, F) — 所有县的时间序列
            adj: (L, L) — 归一化邻接矩阵
            returns: (B, L, horizon)
            """
            B, T, L_loc, F = x.shape

            # Step 1: 每县独立过 LSTM (共享参数)
            x_flat = x.permute(0, 2, 1, 3).reshape(B * L_loc, T, F)  # (B*L, T, F)
            rnn_out, _ = self.backbone.rnn(x_flat)       # (B*L, T, H)
            h_lstm = rnn_out[:, -1, :].reshape(B, L_loc, -1)  # (B, L, H)

            # Step 2: 2 层 GCN 空间传播
            h_gnn = self.gcn1(h_lstm, adj)               # (B, L, gnn_h) — 1-hop 邻居
            h_gnn = self.gcn2(h_gnn, adj)                # (B, L, gnn_h) — 2-hop 邻居

            # Step 3: 融合
            if self.fusion == 'concat':
                h = torch.cat([h_lstm, h_gnn], dim=-1)   # (B, L, H+gnn_h)
            elif self.fusion == 'gate':
                g = torch.sigmoid(self.gate_fc(torch.cat([h_lstm, h_gnn], dim=-1)))  # (B, L, H)
                h = g * h_lstm + (1 - g) * self.gnn_proj(h_gnn)   # (B, L, H)
            else:
                h = h_lstm + self.gnn_proj(h_gnn)

            # Step 4: 预测
            return self.head(h)  # (B, L, horizon)


    def build_adj(locations, k=5):
        """
        构建归一化邻接矩阵 — 基于 Michigan 各县真实地理坐标 (US Census 2020 人口加权质心)。

        策略:
          1. FIPS → 经纬度
          2. Haversine 距离算各县间地理距离
          3. 每个县取 k 个最近邻, 边权重 = exp(-dist / sigma) (高斯核, sigma=邻居距离中位数)
          4. 对称化 + self-loop + D^{-1/2} A D^{-1/2} 归一化 (标准 GCN 归一化)
        """
        # ---- Michigan 83 县质心 (US Census 2020 Centers of Population) ----
        MI_COUNTY_CENTROIDS = {
            "26001": (44.6681, -83.4656), "26003": (46.3845, -86.7118),
            "26005": (42.6084, -85.8710), "26007": (45.0506, -83.5029),
            "26009": (44.9783, -85.1986), "26011": (44.0445, -83.8845),
            "26013": (46.7665, -88.4500), "26015": (42.6185, -85.3494),
            "26017": (43.6300, -83.9292), "26019": (44.6518, -86.0114),
            "26021": (41.9857, -86.4064), "26023": (41.9292, -85.0307),
            "26025": (42.2896, -85.1060), "26027": (41.8923, -86.0405),
            "26029": (45.2445, -85.1110), "26031": (45.5110, -84.5211),
            "26033": (46.3656, -84.3963), "26035": (43.9444, -84.8257),
            "26037": (42.8863, -84.5543), "26039": (44.6571, -84.6690),
            "26041": (45.8019, -87.0658), "26043": (45.8330, -88.0140),
            "26045": (42.6451, -84.7396), "26047": (45.4320, -84.9104),
            "26049": (43.0024, -83.6938), "26051": (43.9680, -84.4398),
            "26053": (46.4464, -89.9930), "26055": (44.7134, -85.6130),
            "26057": (43.3441, -84.6313), "26059": (41.9169, -84.6052),
            "26061": (47.1273, -88.5418), "26063": (43.8324, -83.0630),
            "26065": (42.6884, -84.4863), "26067": (42.9570, -85.0867),
            "26069": (44.3570, -83.5324), "26071": (46.0996, -88.5376),
            "26073": (43.6085, -84.8035), "26075": (42.2418, -84.4066),
            "26077": (42.2625, -85.5864), "26079": (44.7206, -85.1650),
            "26081": (42.9618, -85.6256), "26083": (47.3361, -88.2893),
            "26085": (43.9572, -85.7994), "26087": (43.0582, -83.2494),
            "26089": (44.9070, -85.7301), "26091": (41.9261, -84.0528),
            "26093": (42.5750, -83.8565), "26095": (46.3357, -85.5614),
            "26097": (45.9973, -84.8813), "26099": (42.5891, -82.9579),
            "26101": (44.3063, -86.1853), "26103": (46.4867, -87.4928),
            "26105": (43.9746, -86.3467), "26107": (43.6509, -85.3822),
            "26109": (45.3421, -87.5798), "26111": (43.6433, -84.2964),
            "26113": (44.3077, -85.1799), "26115": (41.9017, -83.4864),
            "26117": (43.2836, -85.1727), "26119": (44.9974, -84.1259),
            "26121": (43.2438, -86.2149), "26123": (43.4662, -85.8113),
            "26125": (42.5869, -83.3140), "26127": (43.6327, -86.3058),
            "26129": (44.2866, -84.1274), "26131": (46.7189, -89.2882),
            "26133": (43.9654, -85.3578), "26135": (44.7015, -84.1484),
            "26137": (45.0069, -84.6668), "26139": (42.9111, -86.0159),
            "26141": (45.3570, -83.8682), "26143": (44.3643, -84.6501),
            "26145": (43.3991, -83.9964), "26147": (42.9122, -82.5635),
            "26149": (41.8875, -85.5269), "26151": (43.3652, -82.7830),
            "26153": (46.0061, -86.2268), "26155": (42.9427, -84.1479),
            "26157": (43.4299, -83.4339), "26159": (42.2602, -86.0013),
            "26161": (42.2576, -83.7269), "26163": (42.3295, -83.2263),
            "26165": (44.3097, -85.4866),
        }

        n = len(locations)
        loc_strs = [str(loc) for loc in locations]

        # 经纬度 → numpy array
        coords = np.array([MI_COUNTY_CENTROIDS[fips] for fips in loc_strs])  # (n, 2)
        lat_rad = np.radians(coords[:, 0])
        lon_rad = np.radians(coords[:, 1])

        # Haversine 距离矩阵 (km)
        dlat = lat_rad[:, None] - lat_rad[None, :]
        dlon = lon_rad[:, None] - lon_rad[None, :]
        a = (np.sin(dlat / 2) ** 2 +
             np.cos(lat_rad[:, None]) * np.cos(lat_rad[None, :]) * np.sin(dlon / 2) ** 2)
        dist = 6371.0 * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))  # (n, n) km

        # KNN: 每个节点取 k 个最近邻 (排除自身)
        A = np.zeros((n, n))
        for i in range(n):
            d_i = dist[i].copy()
            d_i[i] = np.inf
            neighbors = np.argsort(d_i)[:k]
            A[i, neighbors] = 1.0

        # 对称化: i→j 或 j→i 是邻居 → 双向连接
        A = np.maximum(A, A.T)

        # 高斯核权重: w_ij = exp(-dist_ij / sigma), sigma = 邻居距离中位数
        neighbor_dists = dist[A > 0]
        sigma = np.median(neighbor_dists) if len(neighbor_dists) > 0 else 1.0
        W = np.exp(-dist / sigma)
        A = A * W  # 只保留邻居边, 带距离衰减权重

        # 加 self-loop
        A = A + np.eye(n)

        # D^{-1/2} A D^{-1/2} 对称归一化 (标准 GCN)
        D = A.sum(axis=1)
        D_inv_sqrt = np.diag(1.0 / np.sqrt(np.maximum(D, 1e-12)))
        A_norm = D_inv_sqrt @ A @ D_inv_sqrt

        print(f"  build_adj: {n} counties, k={k}, σ={sigma:.1f}km, "
              f"edges={int((A > np.eye(n)).sum())} (excl self-loop)")
        return torch.tensor(A_norm, dtype=torch.float32)


    # ======================== Two-Stage & Ensemble (占位) ========================
    class TwoStageWrapper:
        def __init__(self, cls_model, reg_model, threshold=0.5):
            self.cls, self.reg, self.threshold = cls_model, reg_model, threshold
        def predict(self, x):
            with torch.no_grad():
                return (torch.sigmoid(self.cls(x)) > self.threshold).float() * torch.clamp(self.reg(x), min=0)

    class EnsemblePredictor:
        def __init__(self, method='stacking'):
            self.method = method
            self.preds, self.rmses, self.county_rmses = {}, {}, {}
        def add(self, name, vp, vt, locs):
            self.preds[name] = vp
            cr = {locs[i]: np.sqrt(np.mean((vt[:,i]-vp[:,i])**2)) for i in range(len(locs))}
            self.county_rmses[name] = cr; self.rmses[name] = np.mean(list(cr.values()))
        def fit(self, vt, locs):
            pass
        def predict(self, tp, locs):
            return np.mean(list(tp.values()), axis=0)


    print('✓ GCN / SpatialWrapper / TwoStage / Ensemble 已定义')

[SKIPPED] §3.2 GNN module (SKIP_LSTM_TRAIN=True)


In [ ]:
if SKIP_LSTM_TRAIN:
    print('[SKIPPED] §3.3 model instantiation (SKIP_LSTM_TRAIN=True)')
else:
    # §3.3 自动实例化
    INPUT_DIM = len(feature_cols)

    # ---- 构建 backbone ----
    if MODEL_TYPE == 'simple':
        backbone = SimpleSeq2Seq(INPUT_DIM, HIDDEN_DIM, NUM_LAYERS, HORIZON)
    elif MODEL_TYPE == 'lstm':
        backbone = DeepLSTM(INPUT_DIM, HIDDEN_DIM, NUM_LAYERS, HORIZON,
                            DROPOUT, USE_GRU, BIDIRECTIONAL)
    elif MODEL_TYPE == 'seq2seq':
        backbone = Seq2SeqAttention(INPUT_DIM, HIDDEN_DIM, NUM_LAYERS, HORIZON, DROPOUT)
    elif MODEL_TYPE == 'transformer':
        backbone = TransformerForecaster(INPUT_DIM, HIDDEN_DIM, N_HEADS, NUM_LAYERS, HORIZON,
                                         DIM_FEEDFORWARD, DROPOUT, causal=USE_CAUSAL_MASK)
    else:
        raise ValueError(f'Unknown MODEL_TYPE: {MODEL_TYPE}')

    # ---- 叠加 GNN? ----
    if GNN_ENABLED:
        adj_matrix = build_adj(locations, GNN_K).to(DEVICE)
        model = SpatialWrapper(backbone, HIDDEN_DIM, HORIZON, L,
                               GNN_HIDDEN, SPATIAL_FUSION, DROPOUT).to(DEVICE)
        print(f'Model: {MODEL_TYPE} + GNN ({SPATIAL_FUSION})')
    else:
        model = backbone.to(DEVICE)
        adj_matrix = None
        print(f'Model: {MODEL_TYPE}')

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Params: {n_params:,} | Device: {DEVICE}')
    print(model)

[SKIPPED] §3.3 model instantiation (SKIP_LSTM_TRAIN=True)


---
## §4 损失函数

In [ ]:
if SKIP_LSTM_TRAIN:
    print('[SKIPPED] §4 loss functions (SKIP_LSTM_TRAIN=True)')
else:
    # §4 损失函数
    class WeightedMSE(nn.Module):
        """对大值样本加权的 MSE.
        threshold 是原始空间的值, 如果训练在 log1p 空间自动换算成 log1p(threshold)."""
        def __init__(self, threshold=500, weight=5.0, log_space=False):
            super().__init__()
            if log_space:
                import math
                self.threshold = math.log1p(threshold)  # log1p(500) ≈ 6.22
            else:
                self.threshold = threshold
            self.weight = weight
            print(f'  WeightedMSE: threshold={self.threshold:.3f} (log_space={log_space}), weight={weight}')
        def forward(self, pred, target):
            w = torch.where(target > self.threshold, self.weight, 1.0)
            return (w * (pred - target)**2).mean()

    if LOSS_FN == 'huber':
        criterion = nn.HuberLoss(delta=HUBER_DELTA)
    elif LOSS_FN == 'weighted_mse':
        criterion = WeightedMSE(EXTREME_THRESHOLD, EXTREME_WEIGHT, log_space=LOG_TRANSFORM_Y)
    else:
        criterion = nn.MSELoss()

    # Two-Stage 的分类 loss
    cls_criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([CLS_POS_WEIGHT]).to(DEVICE)) if TWO_STAGE_ENABLED else None

    print(f'Loss: {criterion}')
    if cls_criterion: print(f'CLS Loss: {cls_criterion}')

[SKIPPED] §4 loss functions (SKIP_LSTM_TRAIN=True)


---
## §5 wandb

In [ ]:
if SKIP_LSTM_TRAIN:
    print('[SKIPPED] §5 wandb init (SKIP_LSTM_TRAIN=True)')
else:
    # §5 wandb 初始化
    WANDB_ENABLED = False

    try:
        import wandb

        # ---- 获取配置: 优先 .env, 其次 Colab secrets, 最后手动填 ----
        api_key, entity, user = None, None, 'unknown'

        # 方式 1: .env 文件 (本地)
        try:
            from dotenv import load_dotenv; load_dotenv()
            api_key = os.getenv('WANDB_API_KEY')
            entity  = os.getenv('WANDB_ENTITY')
            user    = os.getenv('WANDB_USERNAME', 'unknown')
        except ImportError:
            pass

        # 方式 2: Colab userdata (Colab 左侧 🔑 Secrets 面板)
        if not api_key and IN_COLAB:
            try:
                from google.colab import userdata
                api_key = userdata.get('WANDB_API_KEY')
                entity  = userdata.get('WANDB_ENTITY')
                user    = userdata.get('WANDB_USERNAME')
            except Exception:
                pass

        # 方式 3: 直接在这里填 (取消注释)
        # api_key = "your_wandb_api_key"
        # entity  = "your_team_name"
        # user    = "your_name"

        if api_key and entity and entity != 'your_team_name':
            wandb.login(key=api_key)
            run_name = f'{user}_{MODEL_NAME}_{EXPERIMENT_TAG}_{datetime.now().strftime("%m%d_%H%M")}'
            wandb.init(project='MLPS-Power-Outage', entity=entity, name=run_name,
                       tags=['phase2', MODEL_TYPE, EXPERIMENT_TAG],
                       config={k: v for k, v in globals().items()
                               if isinstance(v, (int, float, str, bool, list, tuple)) and k.isupper()})
            WANDB_ENABLED = True
            print(f'✓ wandb: {run_name}')
        else:
            print('⚠ wandb 未配置 — 不影响训练, 跳过')
            if IN_COLAB:
                print('  Colab 配置方法: 左侧 🔑 图标 → Add new secret → 添加 WANDB_API_KEY / WANDB_ENTITY')
                print('  或直接在本 cell 取消注释 "方式 3" 手动填入')

    except Exception as e:
        print(f'⚠ wandb 不可用: {e} — 不影响训练, 跳过')

[SKIPPED] §5 wandb init (SKIP_LSTM_TRAIN=True)


---
## §6 训练循环

如果 `ENSEMBLE_ENABLED=True`，跳过此节直接到 §7。

In [ ]:
if SKIP_LSTM_TRAIN:
    print('[SKIPPED] §6.1 optimizer (SKIP_LSTM_TRAIN=True)')
else:
    # §6.1 优化器 + 调度器
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    if LR_SCHEDULER == 'cosine':
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, EPOCHS, eta_min=LEARNING_RATE*0.01)
    elif LR_SCHEDULER == 'step':
        scheduler = optim.lr_scheduler.StepLR(optimizer, 20, 0.5)
    elif LR_SCHEDULER == 'plateau':
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', 0.5, patience=5)
    else:
        scheduler = None
    print(f'Optimizer: AdamW | Scheduler: {LR_SCHEDULER}')

[SKIPPED] §6.1 optimizer (SKIP_LSTM_TRAIN=True)


In [ ]:
if SKIP_LSTM_TRAIN:
    print('[SKIPPED] §6.2 training loop (SKIP_LSTM_TRAIN=True)')
else:
    # §6.2 训练 + 验证 (含 Mixup 数据增强)
    if ENSEMBLE_ENABLED:
        print('ENSEMBLE_ENABLED=True, 跳过训练 → 直接到 §7')
    else:
        best_val_rmse, best_epoch, patience_ctr = float('inf'), 0, 0
        history = {'train_loss':[], 'val_loss':[], 'val_rmse':[], 'lr':[]}
        t_start = time.time()

        print(f'\n{"="*60}')
        print(f'Training: {MODEL_NAME} | {EPOCHS} epochs | {DEVICE}')
        print(f'  Mixup α={MIXUP_ALPHA} | FeatMask={FEAT_MASK_RATIO} | Noise={INPUT_NOISE_STD}')
        print(f'{"="*60}\n')

        for epoch in range(1, EPOCHS + 1):
            # ---- Train ----
            model.train()
            losses = []
            for xb, yb in train_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)

                # ---- Mixup: 随机混合两个样本 ----
                if MIXUP_ALPHA > 0:
                    lam = np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)
                    lam = max(lam, 1 - lam)  # 确保 lam >= 0.5 (保持主样本为主)
                    perm = torch.randperm(xb.size(0), device=xb.device)
                    xb = lam * xb + (1 - lam) * xb[perm]
                    yb = lam * yb + (1 - lam) * yb[perm]

                optimizer.zero_grad()
                pred = model(xb, adj_matrix) if GNN_ENABLED else model(xb)
                if TWO_STAGE_ENABLED and cls_criterion is not None:
                    loss = cls_criterion(pred, (yb > 0).float())
                else:
                    loss = criterion(pred, yb)
                loss.backward()
                if GRADIENT_CLIP: nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
                optimizer.step()
                losses.append(loss.item())
            train_loss = np.mean(losses)

            # ---- Validate ----
            model.eval()
            vl, vp, vt = [], [], []
            with torch.no_grad():
                for xb, yb in val_loader:
                    xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                    pred = model(xb, adj_matrix) if GNN_ENABLED else model(xb)
                    if TWO_STAGE_ENABLED and cls_criterion is not None:
                        vl.append(cls_criterion(pred, (yb>0).float()).item())
                    else:
                        vl.append(criterion(pred, yb).item())
                    vp.append(pred.cpu().numpy()); vt.append(yb.cpu().numpy())
            val_loss = np.mean(vl)
            all_p, all_t = np.concatenate(vp), np.concatenate(vt)
            val_rmse = np.sqrt(np.mean((all_p - all_t)**2))

            # ---- LR schedule ----
            cur_lr = optimizer.param_groups[0]['lr']
            if scheduler:
                scheduler.step(val_rmse) if LR_SCHEDULER == 'plateau' else scheduler.step()

            history['train_loss'].append(train_loss)
            history['val_loss'].append(val_loss)
            history['val_rmse'].append(val_rmse)
            history['lr'].append(cur_lr)

            if WANDB_ENABLED:
                wandb.log({'epoch': epoch, 'train/loss': train_loss,
                           'val/loss': val_loss, 'val/rmse': val_rmse, 'lr': cur_lr})

            if val_rmse < best_val_rmse:
                best_val_rmse, best_epoch, patience_ctr = val_rmse, epoch, 0
                os.makedirs(RESULTS_DIR, exist_ok=True)
                torch.save(model.state_dict(), os.path.join(RESULTS_DIR, f'{MODEL_NAME}_best.pt'))
            else:
                patience_ctr += 1

            if epoch % 5 == 0 or epoch == 1 or patience_ctr == 0:
                m = '★' if patience_ctr == 0 else ' '
                print(f'  E{epoch:3d}/{EPOCHS} | TrLoss:{train_loss:.4f} ValRMSE:{val_rmse:.4f} '
                      f'Best:{best_val_rmse:.4f}@{best_epoch} LR:{cur_lr:.6f} {m}')

            if patience_ctr >= EARLY_STOPPING_PATIENCE:
                print(f'\n  ⏹ Early stop @ epoch {epoch}')
                break

        total_time = time.time() - t_start
        print(f'\n✓ {total_time:.0f}s | Best: {best_val_rmse:.4f} @ epoch {best_epoch}')

[SKIPPED] §6.2 training loop (SKIP_LSTM_TRAIN=True)


In [ ]:
if SKIP_LSTM_TRAIN:
    print('[SKIPPED] §6.3 training curves (SKIP_LSTM_TRAIN=True)')
else:
    # §6.3 训练曲线
    if not ENSEMBLE_ENABLED:
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        axes[0].plot(history['train_loss'], label='Train'); axes[0].plot(history['val_loss'], label='Val')
        axes[0].axvline(best_epoch-1, c='r', ls='--', alpha=.5); axes[0].legend()
        axes[0].set(xlabel='Epoch', ylabel='Loss', yscale='log', title='Loss')
        axes[1].plot(history['val_rmse'], c='#e74c3c')
        axes[1].axhline(best_val_rmse, c='g', ls='--', alpha=.5, label=f'Best={best_val_rmse:.2f}')
        axes[1].legend(); axes[1].set(xlabel='Epoch', ylabel='RMSE', title='Val RMSE')
        axes[2].plot(history['lr'], c='#3498db')
        axes[2].set(xlabel='Epoch', ylabel='LR', yscale='log', title='Learning Rate')
        plt.suptitle(f'{MODEL_NAME} ({EXPERIMENT_TAG})', fontweight='bold'); plt.tight_layout(); plt.show()
        if WANDB_ENABLED: wandb.log({'charts/curves': wandb.Image(fig)})

[SKIPPED] §6.3 training curves (SKIP_LSTM_TRAIN=True)


---
## §7 统一评估 (县均 RMSE)

In [ ]:
# §7.1 评估函数
def evaluate(y_true, y_pred, locs, name='Model'):
    cr = {locs[i]: np.sqrt(np.mean((y_true[:,i]-y_pred[:,i])**2)) for i in range(len(locs))}
    avg = np.mean(list(cr.values()))
    print(f'=== {name}: RMSE={avg:.4f} ===')
    for loc, r in sorted(cr.items(), key=lambda x: -x[1])[:5]:
        print(f'  {loc}: {r:.4f}')
    return avg, cr

def evaluate_horizons(y_true, y_pred, locs, name):
    res = {}
    if y_true.shape[0] >= 24:
        r24, _ = evaluate(y_true[:24], y_pred[:24], locs, f'{name}_24h')
        res[f'{name}_24h'] = r24
    if y_true.shape[0] >= 48:
        r48, _ = evaluate(y_true[:48], y_pred[:48], locs, f'{name}_48h')
        res[f'{name}_48h'] = r48
    rf, cr = evaluate(y_true, y_pred, locs, f'{name}_full')
    res[f'{name}_full'] = rf
    return res, cr

print('✓ evaluate / evaluate_horizons')

✓ evaluate / evaluate_horizons


In [ ]:
if SKIP_LSTM_TRAIN:
    print('[SKIPPED] §7.2 LSTM val inference (SKIP_LSTM_TRAIN=True)')
    import numpy as _np
    Y_pred_val = _np.zeros_like(Y_val_3d_raw)
    print("  stubbed Y_pred_val = zeros (LSTM numbers in §7.3 will be meaningless — ignore)")
else:
    # §7.2 生成验证集预测
    if not ENSEMBLE_ENABLED:
        model.load_state_dict(torch.load(os.path.join(RESULTS_DIR, f'{MODEL_NAME}_best.pt'),
                                         map_location=DEVICE, weights_only=True))
        model.eval()

        Y_pred_val = np.zeros_like(Y_val_3d)
        counts = np.zeros_like(Y_val_3d)

        with torch.no_grad():
            X_full = np.concatenate([X_train_3d[-SEQ_LEN:], X_val_3d], axis=0)

            if GNN_ENABLED:
                for t in range(0, X_full.shape[0]-SEQ_LEN-HORIZON+1, HORIZON):
                    xin = torch.tensor(X_full[t:t+SEQ_LEN], dtype=torch.float32).unsqueeze(0).to(DEVICE)
                    pred = model(xin, adj_matrix).cpu().numpy().squeeze(0)
                    vs = t + SEQ_LEN - SEQ_LEN
                    ve = min(vs + HORIZON, Y_val_3d.shape[0])
                    nf = ve - max(vs, 0)
                    if vs >= 0 and nf > 0:
                        Y_pred_val[vs:ve] += pred[:nf]
                        counts[vs:ve] += 1
            else:
                for li in range(L):
                    for t in range(0, X_full.shape[0]-SEQ_LEN-HORIZON+1, HORIZON):
                        xin = torch.tensor(X_full[t:t+SEQ_LEN, li], dtype=torch.float32).unsqueeze(0).to(DEVICE)
                        pred = model(xin).cpu().numpy().flatten()
                        vs = t + SEQ_LEN - SEQ_LEN
                        ve = min(vs + HORIZON, Y_val_3d.shape[0])
                        nf = ve - max(vs, 0)
                        if vs >= 0 and nf > 0:
                            Y_pred_val[vs:ve, li] += pred[:nf]
                            counts[vs:ve, li] += 1

        counts[counts == 0] = 1
        Y_pred_val = Y_pred_val / counts

        # 根据 target mode 还原到原始空间
        if TARGET_MODE == 'log1p':
            Y_pred_val = np.expm1(np.clip(Y_pred_val, 0, 20))
            print(f'✓ log1p → expm1 还原完成')
        elif TARGET_MODE == 'rate':
            Y_pred_val = np.clip(Y_pred_val, 0, 1) * TRK_val_3d
            print(f'✓ rate × tracked → 原始停电数还原完成')

        Y_pred_val = np.clip(Y_pred_val, 0, None)

        os.makedirs(RESULTS_DIR, exist_ok=True)
        np.save(os.path.join(RESULTS_DIR, f'{MODEL_NAME}_val_pred.npy'), Y_pred_val)
        print(f'Val pred: {Y_pred_val.shape} | range [{Y_pred_val.min():.1f}, {Y_pred_val.max():.1f}]')

[SKIPPED] §7.2 LSTM val inference (SKIP_LSTM_TRAIN=True)
  stubbed Y_pred_val = zeros (LSTM numbers in §7.3 will be meaningless — ignore)


In [ ]:
# §7.3 评估 + 排行榜对比 (统一在"原始停电数空间"比较)

# ---- 在当前 val 上真算 zero / persistence / historical average baseline ----
# Y_pred_val 在 §7.2 已经还原到原始空间 (rate × tracked / expm1)
# 真值也必须用原始空间 Y_val_3d_raw, 否则 rate-mode 下会把所有数字压到 0-1
Y_eval_true = Y_val_3d_raw                          # (T_val, L) 原始停电数, 与 TARGET_MODE 无关
Y_train_raw = df_to_3d(train_df, locations, feature_cols, scaler, 'raw')[1]
T_val, L_val = Y_eval_true.shape

def rmse_horizons(Y_true, Y_pred, name):
    """计算 24h / 48h / full 三个 horizon 的 RMSE (县均)."""
    r = {}
    for h, label in [(24, '24h'), (48, '48h'), (T_val, 'full')]:
        h = min(h, T_val)
        per_county = [np.sqrt(np.mean((Y_true[:h, i] - Y_pred[:h, i])**2)) for i in range(L_val)]
        r[f'{name}_{label}'] = float(np.mean(per_county))
    return r

# Zero baseline (原始空间)
zero_pred = np.zeros_like(Y_eval_true)
zero_res = rmse_horizons(Y_eval_true, zero_pred, 'Zero')

# Persistence: 用 train 最后一个时刻 (原始空间) 作为常值预测
persist_const = Y_train_raw[-1]  # (L,)
persist_pred = np.tile(persist_const, (T_val, 1))
persist_res = rmse_horizons(Y_eval_true, persist_pred, 'Persistence')

# Historical Average: train 每个 county 的均值 (原始空间)
hist_const = Y_train_raw.mean(axis=0)  # (L,)
hist_pred = np.tile(hist_const, (T_val, 1))
hist_res = rmse_horizons(Y_eval_true, hist_pred, 'HistAvg')

print(f'Y_eval_true range: [{Y_eval_true.min():.1f}, {Y_eval_true.max():.1f}] (原始停电数)')
print(f'Y_pred_val  range: [{Y_pred_val.min():.1f}, {Y_pred_val.max():.1f}] (应已在原始空间)')

# ---- 你的模型 ----
if ENSEMBLE_ENABLED:
    print('⚠ Ensemble: 请取消注释加载各模型预测')
    model_results = {}
else:
    model_results, model_county_rmses = evaluate_horizons(Y_eval_true, Y_pred_val, locations, MODEL_NAME)

# 合并 (仅 current-val baselines, 丢弃 old hardcoded)
all_results = {**zero_res, **persist_res, **hist_res, **model_results}

# 把活的 baselines 暴露给后续 cell (§P2.4 要用)
BASELINES_LIVE = {**zero_res, **persist_res, **hist_res}

print(f'\n{"="*70}')
print('RMSE 排行榜 (原始空间, 当前 val)')
print(f'{"="*70}')
for h in ['24h', '48h', 'full']:
    print(f'\n--- {h} ---')
    hr = {k:v for k,v in all_results.items() if k.endswith(f'_{h}')}
    for name, rmse in sorted(hr.items(), key=lambda x: x[1]):
        tag = '  ◀ YOU' if MODEL_NAME in name else ''
        print(f'  {name:25s}: {rmse:10.4f}{tag}')

if WANDB_ENABLED:
    for k,v in model_results.items(): wandb.summary[f'eval/{k}'] = v


Y_eval_true range: [0.0, 11903.0] (原始停电数)
Y_pred_val  range: [0.0, 0.0] (应已在原始空间)
=== DeepLSTM_24h: RMSE=116.1042 ===
  26125: 1929.9280
  26163: 1040.3550
  26161: 948.1747
  26051: 822.5028
  26065: 331.8557
=== DeepLSTM_48h: RMSE=100.4575 ===
  26125: 1824.9698
  26163: 828.0768
  26161: 671.0226
  26051: 581.6004
  26123: 323.6451
=== DeepLSTM_full: RMSE=148.9369 ===
  26163: 1256.2502
  26125: 1164.2473
  26145: 979.1006
  26081: 609.5735
  26103: 586.1528

RMSE 排行榜 (原始空间, 当前 val)

--- 24h ---
  HistAvg_24h              :   113.4185
  Zero_24h                 :   116.1042
  DeepLSTM_24h             :   116.1042  ◀ YOU
  Persistence_24h          :   153.3986

--- 48h ---
  Zero_48h                 :   100.4575
  DeepLSTM_48h             :   100.4575  ◀ YOU
  HistAvg_48h              :   102.0027
  Persistence_48h          :   136.6644

--- full ---
  HistAvg_full             :   148.3560
  Zero_full                :   148.9369
  DeepLSTM_full            :   148.9369  ◀ YOU
  Persis

NameError: name 'WANDB_ENABLED' is not defined

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

FEAT_IDX = {'u10': 84, 'v10': 97, 'gust': 30, 'tp': 82, 'crain': 15}
feat_names = ds.coords['feature'].values.tolist()
for k, i in FEAT_IDX.items():
    assert str(feat_names[i]) == k, f'{k} 期望 idx={i} 实际 {feat_names[i]}'
print('FEAT_IDX 校验通过')

u10 = ds.weather.isel(feature=FEAT_IDX['u10']).values.astype(np.float32)
v10 = ds.weather.isel(feature=FEAT_IDX['v10']).values.astype(np.float32)
gust = ds.weather.isel(feature=FEAT_IDX['gust']).values.astype(np.float32)
tp = ds.weather.isel(feature=FEAT_IDX['tp']).values.astype(np.float32)
crain = ds.weather.isel(feature=FEAT_IDX['crain']).values.astype(np.float32)
ws10 = np.sqrt(u10 ** 2 + v10 ** 2).astype(np.float32)

print(f'  ws10 range=[{ws10.min():.2f}, {ws10.max():.2f}] mean={ws10.mean():.2f}')
print(f'  gust range=[{gust.min():.2f}, {gust.max():.2f}] mean={gust.mean():.2f}')
print(f'  tp   range=[{tp.min():.3f}, {tp.max():.3f}] mean={tp.mean():.4f}')

L, T = ws10.shape
loc_coord = ds['location'].values
ts_coord = pd.to_datetime(ds['timestamp'].values)

df_w = pd.DataFrame({
    'location': np.repeat(loc_coord, T).astype(str),
    'timestamp': np.tile(ts_coord, L),
    'ws10': ws10.flatten(),
    'gust': gust.flatten(),
    'tp': tp.flatten(),
    'crain': crain.flatten(),
}).sort_values(['location', 'timestamp']).reset_index(drop=True)

grp = df_w.groupby('location', sort=False)
df_w['ws10_lag1'] = grp['ws10'].shift(1)
df_w['gust_lag1'] = grp['gust'].shift(1)
df_w['tp_lag1'] = grp['tp'].shift(1)
df_w['crain_lag1'] = grp['crain'].shift(1)

df_w['ws10_delta_3h_causal'] = df_w['ws10_lag1'] - grp['ws10_lag1'].shift(3)
df_w['ws10_delta_6h_causal'] = df_w['ws10_lag1'] - grp['ws10_lag1'].shift(6)
df_w['ws10_rmax_6h_causal'] = grp['ws10_lag1'].transform(lambda s: s.rolling(6, min_periods=1).max())
df_w['ws10_rstd_6h_causal'] = grp['ws10_lag1'].transform(lambda s: s.rolling(6, min_periods=1).std()).fillna(0)
df_w['gust_rmax_6h_causal'] = grp['gust_lag1'].transform(lambda s: s.rolling(6, min_periods=1).max())
df_w['gust_rmax_12h_causal'] = grp['gust_lag1'].transform(lambda s: s.rolling(12, min_periods=1).max())
df_w['tp_sum_6h_causal'] = grp['tp_lag1'].transform(lambda s: s.rolling(6, min_periods=1).sum())
df_w['tp_sum_12h_causal'] = grp['tp_lag1'].transform(lambda s: s.rolling(12, min_periods=1).sum())
df_w['tp_sum_24h_causal'] = grp['tp_lag1'].transform(lambda s: s.rolling(24, min_periods=1).sum())
df_w['crain_any_6h_causal'] = (grp['crain_lag1'].transform(lambda s: s.rolling(6, min_periods=1).max()) > 0).astype(np.float32)

# 州级/全局摘要，帮助模型学“这是不是 statewide storm”
df_w['ws10_state_mean_lag1'] = df_w.groupby('timestamp')['ws10_lag1'].transform('mean')
df_w['ws10_state_max_lag1'] = df_w.groupby('timestamp')['ws10_lag1'].transform('max')
df_w['gust_state_max_lag1'] = df_w.groupby('timestamp')['gust_lag1'].transform('max')
df_w['tp_state_mean_12h'] = df_w.groupby('timestamp')['tp_sum_12h_causal'].transform('mean')
df_w['active_rain_count_lag1'] = df_w.groupby('timestamp')['crain_lag1'].transform(lambda s: (s > 0).sum()).astype(np.float32)

# 本县相对州级偏离
df_w['ws10_minus_state_mean'] = df_w['ws10_lag1'] - df_w['ws10_state_mean_lag1']
df_w['gust_minus_state_max'] = df_w['gust_lag1'] - df_w['gust_state_max_lag1']

v4_weather_cols = [
    'ws10_delta_3h_causal',
    'ws10_delta_6h_causal',
    'ws10_rmax_6h_causal',
    'ws10_rstd_6h_causal',
    'gust_rmax_6h_causal',
    'gust_rmax_12h_causal',
    'tp_sum_6h_causal',
    'tp_sum_12h_causal',
    'tp_sum_24h_causal',
    'crain_any_6h_causal',
    'ws10_state_mean_lag1',
    'ws10_state_max_lag1',
    'gust_state_max_lag1',
    'tp_state_mean_12h',
    'active_rain_count_lag1',
    'ws10_minus_state_mean',
    'gust_minus_state_max',
]
merge_keys = df_w[['location', 'timestamp'] + v4_weather_cols].copy()


def _merge_v4(df_):
    df_ = df_.copy()
    df_['location'] = df_['location'].astype(str)
    df_['timestamp'] = pd.to_datetime(df_['timestamp'])

    extra_cols = v4_weather_cols + [
        'out_state_lag1_mean',
        'out_state_lag3_mean',
        'out_state_active_1h',
        'out_local_vs_state_1h',
    ]
    for c in extra_cols:
        if c in df_.columns:
            df_.drop(columns=[c], inplace=True)

    df_ = df_.merge(merge_keys, on=['location', 'timestamp'], how='left')
    df_['out_state_lag1_mean'] = df_.groupby('timestamp')['out_lag_1h'].transform('mean')
    df_['out_state_lag3_mean'] = df_.groupby('timestamp')['out_lag_3h'].transform('mean')
    df_['out_state_active_1h'] = df_.groupby('timestamp')['out_lag_1h'].transform(lambda s: (s > 0).sum()).astype(np.float32)
    df_['out_local_vs_state_1h'] = df_['out_lag_1h'] / (df_['out_state_lag1_mean'] + 1.0)

    for c in extra_cols:
        df_[c] = df_[c].fillna(0).astype(np.float32)
    return df_


train_df = _merge_v4(train_df)
val_df = _merge_v4(val_df)

v4_cols = v4_weather_cols + [
    'out_state_lag1_mean',
    'out_state_lag3_mean',
    'out_state_active_1h',
    'out_local_vs_state_1h',
]
feature_cols = [c for c in feature_cols if c not in v4_cols] + v4_cols
scaler = StandardScaler().fit(train_df[feature_cols].fillna(0).values)

print()
print(f'v4 新增特征 ({len(v4_cols)}):')
for c in v4_cols:
    print(f'  {c}')
print()
print(f'feature_cols 维度: {len(feature_cols)}')
print(f'train_df 列数: {len(train_df.columns)}  val_df 列数: {len(val_df.columns)}')
print()
print('新特征分布 (train, mean / std / max):')
for c in v4_cols:
    v = train_df[c].values
    print(f'  {c:25s} mean={v.mean():8.3f}  std={v.std():8.3f}  max={v.max():8.2f}')


FEAT_IDX 校验通过
  ws10 range=[0.00, 22.53] mean=3.16
  gust range=[0.00, 29.45] mean=5.98
  tp   range=[0.000, 0.000] mean=0.0000

v4 新增特征 (21):
  ws10_delta_3h_causal
  ws10_delta_6h_causal
  ws10_rmax_6h_causal
  ws10_rstd_6h_causal
  gust_rmax_6h_causal
  gust_rmax_12h_causal
  tp_sum_6h_causal
  tp_sum_12h_causal
  tp_sum_24h_causal
  crain_any_6h_causal
  ws10_state_mean_lag1
  ws10_state_max_lag1
  gust_state_max_lag1
  tp_state_mean_12h
  active_rain_count_lag1
  ws10_minus_state_mean
  gust_minus_state_max
  out_state_lag1_mean
  out_state_lag3_mean
  out_state_active_1h
  out_local_vs_state_1h

feature_cols 维度: 174
train_df 列数: 155  val_df 列数: 155

新特征分布 (train, mean / std / max):
  ws10_delta_3h_causal      mean=  -0.024  std=   1.364  max=   10.08
  ws10_delta_6h_causal      mean=  -0.050  std=   1.836  max=   11.36
  ws10_rmax_6h_causal       mean=   4.091  std=   1.947  max=   19.90
  ws10_rstd_6h_causal       mean=   0.757  std=   0.424  max=    4.86
  gust_rmax_6h_causal  

In [ ]:
# §7.5 HistGBM — 主力模型 ablation (Action 1 + Action 3A) — v3 (pandas pivot)
#
# 修复: 用 pd.pivot + reindex 替代手写 dict lookup, 绕开 numpy scalar hash 问题

import os, time, joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor

os.makedirs('results', exist_ok=True)

print('='*72)
print('HistGBM ablation: baseline / tuned / tuned+weighted (v3)')
print('='*72)

train_sorted = train_df.sort_values(['timestamp', 'location']).reset_index(drop=True)
val_sorted   = val_df.sort_values(['timestamp', 'location']).reset_index(drop=True)

X_tr_tab = scaler.transform(train_sorted[feature_cols].fillna(0).values)
X_va_tab = scaler.transform(val_sorted[feature_cols].fillna(0).values)
y_tr_tab = train_sorted['out'].values.astype(np.float32)
y_tr_log = np.log1p(y_tr_tab)

print(f'  y_tr_tab: n={len(y_tr_tab)}  nonzero%={(y_tr_tab>0).mean()*100:.1f}  max={y_tr_tab.max():.0f}')

# ---- 规范化 location: 统一用 str, 避免 np.str_ vs str 的 hash 歧义 ----
locations_norm = [str(l) for l in locations]
ts_list = sorted(pd.to_datetime(val_df['timestamp']).unique())
ts_list = [pd.Timestamp(t) for t in ts_list]

val_sorted_keys = val_sorted[['timestamp','location']].copy()
val_sorted_keys['timestamp'] = pd.to_datetime(val_sorted_keys['timestamp']).values
val_sorted_keys['location']  = val_sorted_keys['location'].astype(str)

def assemble_3d(pred_1d):
    tmp = val_sorted_keys.copy()
    tmp['pred'] = pred_1d.astype(np.float32)
    # 若有 (timestamp, location) 重复, 取均值
    Y = tmp.pivot_table(index='timestamp', columns='location', values='pred', aggfunc='mean')
    Y = Y.reindex(index=ts_list, columns=locations_norm).fillna(0.0)
    return Y.values.astype(np.float32)

# 健全性: 用全 1 预测, 检查对齐是否完整
_sanity = assemble_3d(np.ones(len(val_sorted_keys), dtype=np.float32))
print(f'  sanity pivot: shape={_sanity.shape} nonzero_cells={(_sanity>0).sum()}/{_sanity.size} ({(_sanity>0).mean()*100:.1f}%)')
assert (_sanity > 0).mean() > 0.95, 'BUG: pivot 后大量 cell 仍为 0, 检查 timestamp/location 原始值'

# ---- Detroit/Wayne 样本权重 ----
DETROIT_WEIGHT = 5.0
DETROIT_KEYS = {'26125', '26163'}
tr_loc_str = train_sorted['location'].astype(str).values
is_detroit = np.isin(tr_loc_str, list(DETROIT_KEYS))
sw_detroit = np.where(is_detroit, DETROIT_WEIGHT, 1.0).astype(np.float32)
print(f'  Detroit/Wayne 样本数: {is_detroit.sum()} / {len(is_detroit)} ({is_detroit.mean()*100:.2f}%), weight={DETROIT_WEIGHT}')
print()

def fit_eval_gbm(name, sample_weight=None, **kwargs):
    print(f'[{name}] training...')
    t0 = time.time()
    gbm = HistGradientBoostingRegressor(
        early_stopping=True, validation_fraction=0.15,
        n_iter_no_change=30, random_state=RANDOM_SEED, **kwargs,
    )
    gbm.fit(X_tr_tab, y_tr_log, sample_weight=sample_weight)
    dt = time.time() - t0
    pred_log = gbm.predict(X_va_tab)
    pred_tab = np.clip(np.expm1(np.clip(pred_log, 0, 20)), 0, None)
    Y_pred = assemble_3d(pred_tab)
    res = rmse_horizons(Y_eval_true, Y_pred, name)
    print(f'  fit={dt:.0f}s  n_iter={gbm.n_iter_}  pred_log[min,max,mean]=[{pred_log.min():.2f},{pred_log.max():.2f},{pred_log.mean():.2f}]')
    print(f'  24h={res[f"{name}_24h"]:.2f}  48h={res[f"{name}_48h"]:.2f}  full={res[f"{name}_full"]:.2f}  pred_max={Y_pred.max():.0f}')
    return gbm, Y_pred, res

gbm_A, YpA, resA = fit_eval_gbm('HistGBM_A_base',
    max_iter=500, learning_rate=0.05, max_depth=8,
    min_samples_leaf=20, l2_regularization=1.0)

gbm_B, YpB, resB = fit_eval_gbm('HistGBM_B_tuned',
    max_iter=2000, learning_rate=0.03, max_depth=None,
    max_leaf_nodes=63, min_samples_leaf=30, l2_regularization=2.0)

gbm_C, YpC, resC = fit_eval_gbm('HistGBM_C_weighted', sample_weight=sw_detroit,
    max_iter=2000, learning_rate=0.03, max_depth=None,
    max_leaf_nodes=63, min_samples_leaf=30, l2_regularization=2.0)

print()
print('='*72)
print(f'{"variant":22s} {"24h":>8s} {"48h":>8s} {"full":>8s} {"Δ24 vs A":>10s}')
print('-'*72)
base24 = resA['HistGBM_A_base_24h']
for tag, res in [('A_base',resA),('B_tuned',resB),('C_weighted',resC)]:
    r24 = res[f'HistGBM_{tag}_24h']; r48 = res[f'HistGBM_{tag}_48h']; rfu = res[f'HistGBM_{tag}_full']
    print(f'HistGBM_{tag:14s} {r24:8.2f} {r48:8.2f} {rfu:8.2f} {r24-base24:+10.2f}')
print('='*72)

variants = {
    'A_base':     (gbm_A, YpA, resA, resA['HistGBM_A_base_24h']),
    'B_tuned':    (gbm_B, YpB, resB, resB['HistGBM_B_tuned_24h']),
    'C_weighted': (gbm_C, YpC, resC, resC['HistGBM_C_weighted_24h']),
}
best_tag = min(variants, key=lambda k: variants[k][3])
best_gbm, best_Yp, best_res, best_24 = variants[best_tag]
print(f'\n最优: HistGBM_{best_tag}  24h={best_24:.2f}')

joblib.dump(best_gbm, 'results/HistGBM_best.pkl')
np.save('results/HistGBM_val_pred.npy', best_Yp)
print(f'  saved: results/HistGBM_best.pkl + results/HistGBM_val_pred.npy shape={best_Yp.shape}')

print(f'\nPer-county 详情 (best variant):')
_ = evaluate_horizons(Y_eval_true, best_Yp, locations, f'HistGBM_{best_tag}')

all_results = {**all_results,
    'HistGBM_24h':  best_res[f'HistGBM_{best_tag}_24h'],
    'HistGBM_48h':  best_res[f'HistGBM_{best_tag}_48h'],
    'HistGBM_full': best_res[f'HistGBM_{best_tag}_full']}

print(f'\n{"="*72}')
print('最终排行榜 (原始空间, interleaved val)')
print(f'{"="*72}')
for h in ['24h','48h','full']:
    print(f'\n--- {h} ---')
    hr = {k:v for k,v in all_results.items() if k.endswith(f'_{h}')}
    for name, rmse in sorted(hr.items(), key=lambda x: x[1]):
        tag = '  ◀ 主力' if name.startswith('HistGBM') else ('  ◀ LSTM' if MODEL_NAME in name else '')
        print(f'  {name:28s}: {rmse:10.4f}{tag}')

print(f'\n{"="*72}')
print('诊断: 是否达标 (<85)?')
print(f'{"="*72}')
if best_24 < 85:
    print(f'  [OK] HistGBM_24h = {best_24:.2f} < 85 → 可以固定主力, 进入 Part II 或做特征工程 v3 进一步压分')
elif best_24 < 92.17:
    print(f'  [~] HistGBM_24h = {best_24:.2f} 比 92.17 好 {92.17-best_24:.2f} 点, 但没到 85')
    print(f'     下一步: 特征工程 v3 (wind_delta / wind_max_6h / spatial_lag)')
else:
    print(f'  [X] HistGBM_24h = {best_24:.2f} ≥ 92.17 → 直接跳特征工程 v3')
print(f'{"="*72}')


HistGBM ablation: baseline / tuned / tuned+weighted (v3)
  y_tr_tab: n=141148  nonzero%=29.5  max=23346
  sanity pivot: shape=(409, 83) nonzero_cells=33034/33947 (97.3%)
  Detroit/Wayne 样本数: 3456 / 141148 (2.45%), weight=5.0

[HistGBM_A_base] training...
  fit=20s  n_iter=194  pred_log[min,max,mean]=[-0.05,8.74,0.71]
  24h=96.76  48h=84.29  full=117.50  pred_max=6259
[HistGBM_B_tuned] training...
  fit=42s  n_iter=326  pred_log[min,max,mean]=[-0.06,8.75,0.71]
  24h=96.39  48h=83.55  full=117.03  pred_max=6290
[HistGBM_C_weighted] training...
  fit=64s  n_iter=470  pred_log[min,max,mean]=[-0.05,8.77,0.71]
  24h=99.33  48h=85.64  full=118.17  pred_max=6419

variant                     24h      48h     full   Δ24 vs A
------------------------------------------------------------------------
HistGBM_A_base            96.76    84.29   117.50      +0.00
HistGBM_B_tuned           96.39    83.55   117.03      -0.37
HistGBM_C_weighted        99.33    85.64   118.17      +2.57

最优: HistGBM_B_tune

In [ ]:
# §7.6 Storm-Tail Ablation — 攻击 Detroit 风暴低估问题
#
# §7.5 的 log1p + MSE 卡 91.84, 风暴县 24h RMSE 高达 1000+.
# 根因: log(y+1) 把 23346 压到 10, 模型在 log 空间 MSE 最优 ≠ 原空间 MSE 最优,
#       HistGBM 会系统性低估极端值.
# 本格测 5 种方案攻 tail, 所有变体共享 v3 特征 + tuned 超参, 只换 target/loss.

import time, numpy as np, joblib, os
from sklearn.ensemble import HistGradientBoostingRegressor

os.makedirs('results', exist_ok=True)

# 复用 §7.5 的 train_sorted, val_sorted, X_tr_tab, X_va_tab, y_tr_tab, assemble_3d, Y_eval_true
# 确保 §7.5 跑过

assert 'X_tr_tab' in dir() and 'assemble_3d' in dir(), '先跑 §7.5 (它定义了 X_tr_tab / assemble_3d)'

y_tr_raw = train_sorted['out'].values.astype(np.float32)
# tracked (rate 变体需要)
trk_tr = train_sorted['tracked'].values.astype(np.float32)
trk_va_long = val_sorted['tracked'].values.astype(np.float32)
trk_va_long = np.where(trk_va_long <= 0, 1.0, trk_va_long)  # 防除零

# tuned 超参 (§7.5 variant B)
TUNED = dict(max_iter=2000, learning_rate=0.03, max_depth=None,
             max_leaf_nodes=63, min_samples_leaf=30, l2_regularization=2.0,
             early_stopping=True, validation_fraction=0.15,
             n_iter_no_change=30, random_state=RANDOM_SEED)

def eval_pred(pred_1d, name):
    Y = assemble_3d(pred_1d.astype(np.float32))
    res = rmse_horizons(Y_eval_true, Y, name)
    print(f'  24h={res[f"{name}_24h"]:7.2f}  48h={res[f"{name}_48h"]:7.2f}  full={res[f"{name}_full"]:7.2f}  pred_max={Y.max():.0f}')
    return Y, res

results_76 = {}
preds_76 = {}

# ---- D1: sqrt target ----
print('[D1] sqrt target, squared_error')
t0 = time.time()
gbm_D1 = HistGradientBoostingRegressor(loss='squared_error', **TUNED)
gbm_D1.fit(X_tr_tab, np.sqrt(y_tr_raw))
pred_sqrt = np.clip(gbm_D1.predict(X_va_tab), 0, None)
pred_D1 = pred_sqrt ** 2
print(f'  fit={time.time()-t0:.0f}s n_iter={gbm_D1.n_iter_}')
Y_D1, r_D1 = eval_pred(pred_D1, 'D1_sqrt')
results_76.update(r_D1); preds_76['D1_sqrt'] = (gbm_D1, Y_D1)

# ---- D2: Poisson loss, raw target ----
print('[D2] Poisson loss, raw y')
t0 = time.time()
try:
    gbm_D2 = HistGradientBoostingRegressor(loss='poisson', **TUNED)
    gbm_D2.fit(X_tr_tab, y_tr_raw)
    pred_D2 = np.clip(gbm_D2.predict(X_va_tab), 0, None)
    print(f'  fit={time.time()-t0:.0f}s n_iter={gbm_D2.n_iter_}')
    Y_D2, r_D2 = eval_pred(pred_D2, 'D2_poisson')
    results_76.update(r_D2); preds_76['D2_poisson'] = (gbm_D2, Y_D2)
except Exception as e:
    print(f'  SKIP: {e}')

# ---- D3: Quantile 0.85 (raw target) ----
print('[D3] quantile loss, alpha=0.85, raw y')
t0 = time.time()
try:
    gbm_D3 = HistGradientBoostingRegressor(loss='quantile', quantile=0.85, **TUNED)
    gbm_D3.fit(X_tr_tab, y_tr_raw)
    pred_D3 = np.clip(gbm_D3.predict(X_va_tab), 0, None)
    print(f'  fit={time.time()-t0:.0f}s n_iter={gbm_D3.n_iter_}')
    Y_D3, r_D3 = eval_pred(pred_D3, 'D3_quantile85')
    results_76.update(r_D3); preds_76['D3_quantile85'] = (gbm_D3, Y_D3)
except Exception as e:
    print(f'  SKIP: {e}')

# ---- D4: log1p + magnitude weight ----
print('[D4] log1p target, sample_weight = 1 + y/200')
t0 = time.time()
sw_mag = (1.0 + y_tr_raw / 200.0).astype(np.float32)
print(f'  weight: min={sw_mag.min():.2f} max={sw_mag.max():.2f} mean={sw_mag.mean():.2f} p99={np.percentile(sw_mag,99):.2f}')
gbm_D4 = HistGradientBoostingRegressor(loss='squared_error', **TUNED)
gbm_D4.fit(X_tr_tab, np.log1p(y_tr_raw), sample_weight=sw_mag)
pred_D4 = np.clip(np.expm1(np.clip(gbm_D4.predict(X_va_tab), 0, 20)), 0, None)
print(f'  fit={time.time()-t0:.0f}s n_iter={gbm_D4.n_iter_}')
Y_D4, r_D4 = eval_pred(pred_D4, 'D4_magweight')
results_76.update(r_D4); preds_76['D4_magweight'] = (gbm_D4, Y_D4)

# ---- D5: rate target (y/tracked), log1p ----
print('[D5] rate target log1p(y/tracked), inference * tracked')
t0 = time.time()
y_rate = y_tr_raw / np.where(trk_tr <= 0, 1.0, trk_tr)
gbm_D5 = HistGradientBoostingRegressor(loss='squared_error', **TUNED)
gbm_D5.fit(X_tr_tab, np.log1p(y_rate))
pred_rate_log = gbm_D5.predict(X_va_tab)
pred_rate = np.clip(np.expm1(np.clip(pred_rate_log, 0, 20)), 0, None)
pred_D5 = pred_rate * trk_va_long  # 回到原空间
print(f'  fit={time.time()-t0:.0f}s n_iter={gbm_D5.n_iter_}  rate_max={pred_rate.max():.4f}  raw_max={pred_D5.max():.0f}')
Y_D5, r_D5 = eval_pred(pred_D5, 'D5_rate')
results_76.update(r_D5); preds_76['D5_rate'] = (gbm_D5, Y_D5)

# ---- 汇总对比 §7.5 baseline ----
print()
print('='*78)
print(f'{"variant":22s} {"24h":>8s} {"48h":>8s} {"full":>8s} {"Δ24 vs A_base":>14s}')
print('-'*78)
base = all_results.get('HistGBM_24h', 91.84)  # §7.5 best
print(f'{"§7.5 HistGBM (ref)":22s} {base:8.2f} {"--":>8s} {"--":>8s} {"(reference)":>14s}')
ordered = ['D1_sqrt','D2_poisson','D3_quantile85','D4_magweight','D5_rate']
for tag in ordered:
    k24 = f'{tag}_24h'
    if k24 not in results_76: continue
    r24 = results_76[k24]
    r48 = results_76.get(f'{tag}_48h', float('nan'))
    rfu = results_76.get(f'{tag}_full', float('nan'))
    print(f'{tag:22s} {r24:8.2f} {r48:8.2f} {rfu:8.2f} {r24-base:+14.2f}')
print('='*78)

# ---- 选最好 + per-county top-5 + 保存 ----
best_tag = min((t for t in ordered if f'{t}_24h' in results_76), key=lambda t: results_76[f'{t}_24h'])
best_gbm, best_Y = preds_76[best_tag]
best_24 = results_76[f'{best_tag}_24h']
print()
print(f'最优 tail-attack 变体: {best_tag}  24h={best_24:.2f}')
print(f'  vs §7.5 baseline: {best_24-base:+.2f}')

if best_24 < base:
    print(f'  → 比 §7.5 好, 覆盖 results/HistGBM_best.pkl')
    joblib.dump(best_gbm, 'results/HistGBM_best.pkl')
    np.save('results/HistGBM_val_pred.npy', best_Y)
    all_results['HistGBM_24h']  = best_24
    all_results['HistGBM_48h']  = results_76[f'{best_tag}_48h']
    all_results['HistGBM_full'] = results_76[f'{best_tag}_full']
else:
    print(f'  → 没超过 §7.5, 保持原 best.pkl 不动')

print()
print(f'Per-county 详情 ({best_tag}):')
_ = evaluate_horizons(Y_eval_true, best_Y, locations, best_tag)

print()
print('='*78)
print('诊断')
print('='*78)
if best_24 < 85:
    print(f'  [OK] {best_tag} = {best_24:.2f} < 85 目标达成')
elif best_24 < base:
    print(f'  [~] {best_tag} = {best_24:.2f}, 比 baseline 省 {base-best_24:.2f} 点, 但未到 85')
    print(f'     下一步: 考虑 ensemble 或 two-stage (storm classifier + conditional regressor)')
else:
    print(f'  [X] 所有 tail-attack 方案都没超过 log1p+MSE 的 {base:.2f}')
    print(f'     下一步: 接受 91.84 为 Part I 最终, 转 Part II')
print('='*78)


[D1] sqrt target, squared_error
  fit=32s n_iter=288
  24h=  93.45  48h=  81.09  full= 106.41  pred_max=6105
[D2] Poisson loss, raw y
  fit=6s n_iter=30
  24h= 107.35  48h=  93.09  full= 117.59  pred_max=8975
[D3] quantile loss, alpha=0.85, raw y
  fit=51s n_iter=261
  24h= 116.69  48h=  98.55  full= 113.69  pred_max=8043
[D4] log1p target, sample_weight = 1 + y/200
  weight: min=1.00 max=117.73 mean=1.23 p99=5.63
  fit=33s n_iter=202
  24h=  96.37  48h=  82.60  full= 103.88  pred_max=5951
[D5] rate target log1p(y/tracked), inference * tracked
  fit=20s n_iter=176  rate_max=0.1511  raw_max=10240
  24h= 123.03  48h= 100.00  full= 111.62  pred_max=10240

variant                     24h      48h     full  Δ24 vs A_base
------------------------------------------------------------------------------
§7.5 HistGBM (ref)        96.39       --       --    (reference)
D1_sqrt                   93.45    81.09   106.41          -2.94
D2_poisson               107.35    93.09   117.59         +10.96


In [ ]:
# §7.7 Post-sqrt refinement — combo / exponent / ensemble
#
# §7.6 的 D1 sqrt = 88.19 确认 target 变换是主杠杆.
# 这里在 sqrt 骨架上叠加策略, 外加 exponent 搜索 + ensemble.

import time, numpy as np, joblib, os
from sklearn.ensemble import HistGradientBoostingRegressor

assert 'X_tr_tab' in dir() and 'preds_76' in dir(), '先跑 §7.5 + §7.6'

y_tr_raw = train_sorted['out'].values.astype(np.float32)
TUNED = dict(max_iter=2000, learning_rate=0.03, max_depth=None,
             max_leaf_nodes=63, min_samples_leaf=30, l2_regularization=2.0,
             early_stopping=True, validation_fraction=0.15,
             n_iter_no_change=30, random_state=RANDOM_SEED)

def eval_pred(pred_1d, name):
    Y = assemble_3d(pred_1d.astype(np.float32))
    res = rmse_horizons(Y_eval_true, Y, name)
    print(f'  24h={res[f"{name}_24h"]:7.2f}  48h={res[f"{name}_48h"]:7.2f}  full={res[f"{name}_full"]:7.2f}  pred_max={Y.max():.0f}')
    return Y, res

results_77 = {}
preds_77 = {}

# ---- E1: sqrt + magnitude weight ----
print('[E1] sqrt target + sample_weight = 1 + y/200')
t0 = time.time()
sw_mag = (1.0 + y_tr_raw / 200.0).astype(np.float32)
gbm_E1 = HistGradientBoostingRegressor(loss='squared_error', **TUNED)
gbm_E1.fit(X_tr_tab, np.sqrt(y_tr_raw), sample_weight=sw_mag)
pred_E1 = np.clip(gbm_E1.predict(X_va_tab), 0, None) ** 2
print(f'  fit={time.time()-t0:.0f}s n_iter={gbm_E1.n_iter_}')
Y_E1, r_E1 = eval_pred(pred_E1, 'E1_sqrt_magw')
results_77.update(r_E1); preds_77['E1_sqrt_magw'] = (gbm_E1, Y_E1)

# ---- E2: y^0.4 ----
print('[E2] y^0.4 target')
t0 = time.time()
P = 0.4
gbm_E2 = HistGradientBoostingRegressor(loss='squared_error', **TUNED)
gbm_E2.fit(X_tr_tab, np.power(y_tr_raw, P))
pred_E2 = np.clip(gbm_E2.predict(X_va_tab), 0, None) ** (1.0/P)
print(f'  fit={time.time()-t0:.0f}s n_iter={gbm_E2.n_iter_}')
Y_E2, r_E2 = eval_pred(pred_E2, 'E2_p04')
results_77.update(r_E2); preds_77['E2_p04'] = (gbm_E2, Y_E2)

# ---- E3: ensemble avg(D1_sqrt, A_base log1p) ----
# 需要重新训 A_base (log1p) 因为 §7.5 的 gbm 在 §7.6 里没存指针, 只存了 best
print('[E3] ensemble: avg(D1_sqrt, log1p_A_base)')
t0 = time.time()
gbm_log = HistGradientBoostingRegressor(loss='squared_error',
    max_iter=500, learning_rate=0.05, max_depth=8,
    min_samples_leaf=20, l2_regularization=1.0,
    early_stopping=True, validation_fraction=0.15,
    n_iter_no_change=20, random_state=RANDOM_SEED)
gbm_log.fit(X_tr_tab, np.log1p(y_tr_raw))
pred_log_raw = np.clip(np.expm1(np.clip(gbm_log.predict(X_va_tab), 0, 20)), 0, None)

# D1 pred from §7.6
_, Y_D1 = preds_76['D1_sqrt']
# 重新取 D1 的 1d 预测: 从 3D 反推不容易, 直接用 D1 gbm.predict
gbm_D1_ref = preds_76['D1_sqrt'][0]
pred_D1_raw = np.clip(gbm_D1_ref.predict(X_va_tab), 0, None) ** 2

pred_E3 = 0.5 * pred_log_raw + 0.5 * pred_D1_raw
print(f'  fit={time.time()-t0:.0f}s')
Y_E3, r_E3 = eval_pred(pred_E3, 'E3_ens_avg')
results_77.update(r_E3); preds_77['E3_ens_avg'] = (gbm_log, Y_E3)  # 占位 model

# ---- E4: weighted ensemble 0.7 sqrt + 0.3 log1p ----
pred_E4 = 0.7 * pred_D1_raw + 0.3 * pred_log_raw
Y_E4, r_E4 = eval_pred(pred_E4, 'E4_ens_07sqrt')
results_77.update(r_E4); preds_77['E4_ens_07sqrt'] = (None, Y_E4)

# ---- E5: weighted ensemble 0.3 sqrt + 0.7 log1p ----
pred_E5 = 0.3 * pred_D1_raw + 0.7 * pred_log_raw
Y_E5, r_E5 = eval_pred(pred_E5, 'E5_ens_03sqrt')
results_77.update(r_E5); preds_77['E5_ens_03sqrt'] = (None, Y_E5)

# ---- 汇总 ----
print()
print('='*78)
print(f'{"variant":22s} {"24h":>8s} {"48h":>8s} {"full":>8s} {"Δ vs D1_sqrt":>14s}')
print('-'*78)
ref = results_76.get('D1_sqrt_24h', 88.19)
print(f'{"§7.6 D1_sqrt (ref)":22s} {ref:8.2f} {"--":>8s} {"--":>8s} {"(ref)":>14s}')
ordered = ['E1_sqrt_magw','E2_p04','E3_ens_avg','E4_ens_07sqrt','E5_ens_03sqrt']
for tag in ordered:
    k = f'{tag}_24h'
    if k not in results_77: continue
    r24 = results_77[k]
    r48 = results_77.get(f'{tag}_48h', float('nan'))
    rfu = results_77.get(f'{tag}_full', float('nan'))
    print(f'{tag:22s} {r24:8.2f} {r48:8.2f} {rfu:8.2f} {r24-ref:+14.2f}')
print('='*78)

best_tag = min((t for t in ordered if f'{t}_24h' in results_77), key=lambda t: results_77[f'{t}_24h'])
best_24 = results_77[f'{best_tag}_24h']
print(f'\n最优 post-sqrt 变体: {best_tag}  24h={best_24:.2f}')
print(f'  vs D1 sqrt: {best_24-ref:+.2f}')

# 与全局 best 比较
global_best = min(all_results['HistGBM_24h'], ref, best_24)
if best_24 < all_results['HistGBM_24h']:
    print(f'  → 新 global best, 覆盖 results/HistGBM_best.pkl')
    _, Y_best = preds_77[best_tag]
    np.save('results/HistGBM_val_pred.npy', Y_best)
    all_results['HistGBM_24h']  = best_24
    all_results['HistGBM_48h']  = results_77[f'{best_tag}_48h']
    all_results['HistGBM_full'] = results_77[f'{best_tag}_full']
    if preds_77[best_tag][0] is not None:
        joblib.dump(preds_77[best_tag][0], 'results/HistGBM_best.pkl')

print()
print(f'Per-county 详情 ({best_tag}):')
_ = evaluate_horizons(Y_eval_true, preds_77[best_tag][1], locations, best_tag)

print()
print('='*78)
if best_24 < 85:
    print(f'[OK] {best_tag} = {best_24:.2f} < 85 目标达成')
elif best_24 < ref:
    print(f'[~] 省 {ref-best_24:.2f} 点, 仍未到 85')
    print(f'    下一步: 考虑 two-stage (storm classifier + conditional)')
else:
    print(f'[X] 没超过 D1 sqrt, 接受 88.19 为 Part I 最终')
print('='*78)


[E1] sqrt target + sample_weight = 1 + y/200
  fit=32s n_iter=228
  24h= 126.56  48h= 103.81  full= 111.69  pred_max=7365
[E2] y^0.4 target
  fit=24s n_iter=186
  24h=  93.48  48h=  80.90  full= 107.92  pred_max=5661
[E3] ensemble: avg(D1_sqrt, log1p_A_base)
  fit=21s
  24h=  93.42  48h=  81.28  full= 110.60  pred_max=6029
  24h=  93.06  48h=  80.88  full= 108.54  pred_max=5932
  24h=  94.32  48h=  82.12  full= 113.13  pred_max=6125

variant                     24h      48h     full   Δ vs D1_sqrt
------------------------------------------------------------------------------
§7.6 D1_sqrt (ref)        93.45       --       --          (ref)
E1_sqrt_magw             126.56   103.81   111.69         +33.11
E2_p04                    93.48    80.90   107.92          +0.03
E3_ens_avg                93.42    81.28   110.60          -0.03
E4_ens_07sqrt             93.06    80.88   108.54          -0.40
E5_ens_03sqrt             94.32    82.12   113.13          +0.87

最优 post-sqrt 变体: E4_ens_07s

In [ ]:
# §7.9 County-scale target + county static stats + blend sweep
#
# 目标:
# 1) 不用 county one-hot, 改用平滑的县级静态统计
# 2) 不用 tracked-rate, 改用 county-scale target
# 3) 对 scaled-sqrt / log1p 做 alpha sweep

import os, time, joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor

assert 'X_tr_tab' in dir() and 'X_va_tab' in dir(), '先跑到 §7.5'
assert 'train_sorted' in dir() and 'val_sorted' in dir(), '先跑 §7.5'
assert 'assemble_3d' in dir() and 'Y_eval_true' in dir(), '先跑 §7.5'
assert 'all_results' in dir(), '先跑 §7.5/§7.7'

y_tr_raw = train_sorted['out'].values.astype(np.float32)
tr_loc = train_sorted['location'].astype(str).values
va_loc = val_sorted['location'].astype(str).values

TUNED = dict(
    max_iter=2000,
    learning_rate=0.03,
    max_depth=None,
    max_leaf_nodes=63,
    min_samples_leaf=30,
    l2_regularization=2.0,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=30,
    random_state=RANDOM_SEED,
)

def eval_pred(pred_1d, name):
    Y = assemble_3d(pred_1d.astype(np.float32))
    res = rmse_horizons(Y_eval_true, Y, name)
    print(f'  24h={res[f"{name}_24h"]:7.2f}  48h={res[f"{name}_48h"]:7.2f}  full={res[f"{name}_full"]:7.2f}  pred_max={Y.max():.0f}')
    return Y, res

# ============================================================================
# G0: train-only county static stats
# ============================================================================
print('[G0] build county static stats (train-only)')

tmp = train_sorted[['location', 'out', 'tracked']].copy()
tmp['location'] = tmp['location'].astype(str)

def pos_mean(s):
    s = s[s > 0]
    return float(s.mean()) if len(s) else 0.0

cty_stats = tmp.groupby('location').agg(
    cty_out_mean=('out', 'mean'),
    cty_out_std=('out', 'std'),
    cty_out_p90=('out', lambda s: float(np.percentile(s, 90))),
    cty_out_p95=('out', lambda s: float(np.percentile(s, 95))),
    cty_nonzero_rate=('out', lambda s: float((s > 0).mean())),
    cty_tracked_mean=('tracked', 'mean'),
).astype(np.float32)

cty_pos_mean = tmp.groupby('location')['out'].apply(pos_mean).astype(np.float32)
cty_stats['cty_out_pos_mean'] = cty_pos_mean
cty_stats['cty_out_std'] = cty_stats['cty_out_std'].fillna(0).astype(np.float32)

# 温和 scale: 不是 tracked-rate，也不是 raw y
cty_stats['cty_scale'] = np.maximum(
    25.0,
    0.5 * cty_stats['cty_out_pos_mean'].values + 0.5 * cty_stats['cty_out_p95'].values
).astype(np.float32)

stat_cols = [
    'cty_out_mean',
    'cty_out_std',
    'cty_out_p90',
    'cty_out_p95',
    'cty_nonzero_rate',
    'cty_tracked_mean',
    'cty_out_pos_mean',
]

# 把 county stats 映射到每条样本
stat_tr_raw = np.column_stack([
    pd.Series(tr_loc).map(cty_stats[c]).fillna(cty_stats[c].mean()).values.astype(np.float32)
    for c in stat_cols
])
stat_va_raw = np.column_stack([
    pd.Series(va_loc).map(cty_stats[c]).fillna(cty_stats[c].mean()).values.astype(np.float32)
    for c in stat_cols
])

# 简单 z-score，避免这些静态特征量纲太散
mu = stat_tr_raw.mean(axis=0, keepdims=True)
sd = stat_tr_raw.std(axis=0, keepdims=True)
sd[sd == 0] = 1.0
stat_tr = (stat_tr_raw - mu) / sd
stat_va = (stat_va_raw - mu) / sd

X_tr_G = np.hstack([X_tr_tab, stat_tr]).astype(np.float32)
X_va_G = np.hstack([X_va_tab, stat_va]).astype(np.float32)

scale_tr = pd.Series(tr_loc).map(cty_stats['cty_scale']).fillna(cty_stats['cty_scale'].mean()).values.astype(np.float32)
scale_va = pd.Series(va_loc).map(cty_stats['cty_scale']).fillna(cty_stats['cty_scale'].mean()).values.astype(np.float32)

print(f'  X_tr_G shape: {X_tr_G.shape}  (原 {X_tr_tab.shape[1]} + {len(stat_cols)} county stats)')
print(f'  county scale: min={scale_tr.min():.1f} mean={scale_tr.mean():.1f} p95={np.percentile(scale_tr,95):.1f} max={scale_tr.max():.1f}')

results_79 = {}
preds_79 = {}

# ============================================================================
# G1: county-scale sqrt target
# ============================================================================
print('\n[G1] county-scale sqrt target + county stats')
t0 = time.time()

# y / county_scale -> sqrt
y_scaled_sqrt = np.sqrt(y_tr_raw / np.maximum(scale_tr, 1.0))

gbm_G1 = HistGradientBoostingRegressor(loss='squared_error', **TUNED)
gbm_G1.fit(X_tr_G, y_scaled_sqrt)

pred_scaled_sqrt = np.clip(gbm_G1.predict(X_va_G), 0, None)
pred_G1 = (pred_scaled_sqrt ** 2) * scale_va

print(f'  fit={time.time()-t0:.0f}s n_iter={gbm_G1.n_iter_}')
Y_G1, r_G1 = eval_pred(pred_G1, 'G1_cty_scale_sqrt')
results_79.update(r_G1)
preds_79['G1_cty_scale_sqrt'] = (gbm_G1, Y_G1, pred_G1)

# ============================================================================
# G2: log1p raw target + county stats
# ============================================================================
print('\n[G2] log1p raw target + county stats')
t0 = time.time()

gbm_G2 = HistGradientBoostingRegressor(loss='squared_error', **TUNED)
gbm_G2.fit(X_tr_G, np.log1p(y_tr_raw))

pred_G2 = np.clip(np.expm1(np.clip(gbm_G2.predict(X_va_G), 0, 20)), 0, None)

print(f'  fit={time.time()-t0:.0f}s n_iter={gbm_G2.n_iter_}')
Y_G2, r_G2 = eval_pred(pred_G2, 'G2_log_county')
results_79.update(r_G2)
preds_79['G2_log_county'] = (gbm_G2, Y_G2, pred_G2)

# ============================================================================
# G3: alpha sweep blend
# ============================================================================
print('\n[G3] alpha sweep: alpha * G1_scaled_sqrt + (1-alpha) * G2_log')
alphas = np.arange(0.50, 0.91, 0.05)

best_alpha = None
best_alpha_24 = float('inf')
best_alpha_pred = None
best_alpha_res = None
best_alpha_Y = None

for a in alphas:
    pred_blend = a * pred_G1 + (1.0 - a) * pred_G2
    Y_blend, r_blend = eval_pred(pred_blend, f'G3_blend_{a:.2f}')
    results_79.update(r_blend)
    preds_79[f'G3_blend_{a:.2f}'] = (None, Y_blend, pred_blend)

    cur24 = r_blend[f'G3_blend_{a:.2f}_24h']
    if cur24 < best_alpha_24:
        best_alpha = a
        best_alpha_24 = cur24
        best_alpha_pred = pred_blend.copy()
        best_alpha_res = r_blend
        best_alpha_Y = Y_blend

# ============================================================================
# 汇总
# ============================================================================
print()
print('=' * 86)
print(f'{"variant":24s} {"24h":>8s} {"48h":>8s} {"full":>8s} {"Δ vs current best":>18s}')
print('-' * 86)

ref = all_results.get('HistGBM_24h', 999.0)
print(f'{"current global best":24s} {ref:8.2f} {"--":>8s} {"--":>8s} {"(ref)":>18s}')

ordered = ['G1_cty_scale_sqrt', 'G2_log_county']
for tag in ordered:
    r24 = results_79[f'{tag}_24h']
    r48 = results_79[f'{tag}_48h']
    rfu = results_79[f'{tag}_full']
    print(f'{tag:24s} {r24:8.2f} {r48:8.2f} {rfu:8.2f} {r24-ref:+18.2f}')

print(f'{"best G3 alpha":24s} {best_alpha_24:8.2f} {best_alpha_res[f"G3_blend_{best_alpha:.2f}_48h"]:8.2f} {best_alpha_res[f"G3_blend_{best_alpha:.2f}_full"]:8.2f} {best_alpha_24-ref:+18.2f}')
print('=' * 86)

print(f'\n最优 G3 blend: alpha={best_alpha:.2f}  24h={best_alpha_24:.2f}')

print(f'\nPer-county (best G3 alpha={best_alpha:.2f}):')
_ = evaluate_horizons(Y_eval_true, best_alpha_Y, locations, f'G3_blend_{best_alpha:.2f}')

# 可选保存
if best_alpha_24 < ref:
    print(f'\n  → 新 global best ({ref:.2f} -> {best_alpha_24:.2f}), 保存到 results/')
    np.save('results/HistGBM_val_pred.npy', best_alpha_Y)
    joblib.dump(
        {
            'variant': 'G3_county_scale_blend',
            'alpha': float(best_alpha),
            'model_scaled_sqrt': gbm_G1,
            'model_log_county': gbm_G2,
            'county_stats': cty_stats,
            'stat_cols': stat_cols,
            'stat_mu': mu,
            'stat_sd': sd,
        },
        'results/HistGBM_best.pkl'
    )
    all_results['HistGBM_24h'] = best_alpha_24
    all_results['HistGBM_48h'] = best_alpha_res[f'G3_blend_{best_alpha:.2f}_48h']
    all_results['HistGBM_full'] = best_alpha_res[f'G3_blend_{best_alpha:.2f}_full']
else:
    print(f'\n  → 没超过 current best={ref:.2f}, 不覆盖 results/')

print('\n' + '=' * 86)
if best_alpha_24 < 90:
    print(f'[OK] G3 alpha={best_alpha:.2f} 把 24h 压到 {best_alpha_24:.2f}，值得保留')
elif best_alpha_24 < ref:
    print(f'[~] 比 current best 省了 {ref - best_alpha_24:.2f} 点，可以保留')
else:
    print(f'[X] county-scale 这条线没带来提升，基本可以收手转 Part II')
print('=' * 86)


[G0] build county static stats (train-only)
  X_tr_G shape: (141148, 181)  (原 174 + 7 county stats)
  county scale: min=25.0 mean=138.2 p95=308.1 max=2465.9

[G1] county-scale sqrt target + county stats
  fit=18s n_iter=128
  24h= 100.96  48h=  85.36  full= 108.51  pred_max=7579

[G2] log1p raw target + county stats
  fit=44s n_iter=322
  24h=  97.77  48h=  84.23  full= 117.46  pred_max=7418

[G3] alpha sweep: alpha * G1_scaled_sqrt + (1-alpha) * G2_log
  24h=  95.03  48h=  81.29  full= 110.63  pred_max=6022
  24h=  95.27  48h=  81.41  full= 110.21  pred_max=5883
  24h=  95.60  48h=  81.60  full= 109.84  pred_max=5743
  24h=  96.02  48h=  81.86  full= 109.51  pred_max=5762
  24h=  96.52  48h=  82.19  full= 109.23  pred_max=6022
  24h=  97.10  48h=  82.58  full= 109.00  pred_max=6281
  24h=  97.74  48h=  83.03  full= 108.81  pred_max=6541
  24h=  98.45  48h=  83.53  full= 108.67  pred_max=6800
  24h=  99.23  48h=  84.09  full= 108.57  pred_max=7060

variant                       24h    

In [ ]:
# §7.8 County-level specialization — F1 one-hot + F2 per-county + F3 storm-specialist
#
# §7.7 卡在 88.14, 4 个大县 (26125/26163/26161/26051) 主导误差.
# 三路攻:
#   F1: 加 county one-hot 让模型学县级偏置
#   F2: 为 4 个大县各训专属 sqrt HistGBM, 替换那些县的预测
#   F3: 仅在 out>100 样本上训 storm-specialist, 用分类器 gate 它
#
# 依赖 §7.5/§7.6 的产物: X_tr_tab, X_va_tab, y_tr_raw, assemble_3d, preds_76, preds_77

import time, numpy as np, joblib, os, pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier

assert 'X_tr_tab' in dir() and 'preds_77' in dir(), '先跑 §7.5/§7.6/§7.7'

y_tr_raw = train_sorted['out'].values.astype(np.float32)
tr_loc   = train_sorted['location'].astype(str).values
va_loc   = val_sorted['location'].astype(str).values

TUNED = dict(max_iter=2000, learning_rate=0.03, max_depth=None,
             max_leaf_nodes=63, min_samples_leaf=30, l2_regularization=2.0,
             early_stopping=True, validation_fraction=0.15,
             n_iter_no_change=30, random_state=RANDOM_SEED)

def eval_pred(pred_1d, name):
    Y = assemble_3d(pred_1d.astype(np.float32))
    res = rmse_horizons(Y_eval_true, Y, name)
    print(f'  24h={res[f"{name}_24h"]:7.2f}  48h={res[f"{name}_48h"]:7.2f}  full={res[f"{name}_full"]:7.2f}  pred_max={Y.max():.0f}')
    return Y, res

results_78 = {}
preds_78 = {}

# ============================================================================
# F1: 加 county one-hot, 重训 sqrt
# ============================================================================
print('[F1] county one-hot + sqrt target')
t0 = time.time()
all_locs = sorted(set(tr_loc.tolist() + va_loc.tolist()))
loc2onehot = {l: i for i, l in enumerate(all_locs)}
oh_tr = np.zeros((len(tr_loc), len(all_locs)), dtype=np.float32)
oh_va = np.zeros((len(va_loc), len(all_locs)), dtype=np.float32)
oh_tr[np.arange(len(tr_loc)), [loc2onehot[l] for l in tr_loc]] = 1
oh_va[np.arange(len(va_loc)), [loc2onehot[l] for l in va_loc]] = 1

X_tr_F1 = np.hstack([X_tr_tab, oh_tr])
X_va_F1 = np.hstack([X_va_tab, oh_va])
print(f'  X_tr_F1 shape: {X_tr_F1.shape}  (原 {X_tr_tab.shape[1]} + {len(all_locs)} one-hot)')

gbm_F1 = HistGradientBoostingRegressor(loss='squared_error', **TUNED)
gbm_F1.fit(X_tr_F1, np.sqrt(y_tr_raw))
pred_F1 = np.clip(gbm_F1.predict(X_va_F1), 0, None) ** 2
print(f'  fit={time.time()-t0:.0f}s n_iter={gbm_F1.n_iter_}')
Y_F1, r_F1 = eval_pred(pred_F1, 'F1_onehot_sqrt')
results_78.update(r_F1); preds_78['F1_onehot_sqrt'] = (gbm_F1, Y_F1, pred_F1)

# ============================================================================
# F2: 4 个大县专属模型, 替换这些县的预测 (基于 F1 best 或 D1 全局)
# ============================================================================
print('\n[F2] per-county specialists for 4 worst counties')
WORST_COUNTIES = ['26125', '26163', '26161', '26051']  # Detroit / Wayne / Washtenaw / Gratiot

# 用 F1 (目前最强) 作 base, 给专属模型替换 4 县
base_pred_1d = pred_F1.copy()  # 从 F1 拷贝
_, Y_D1 = preds_76['D1_sqrt']
gbm_D1 = preds_76['D1_sqrt'][0]
base_pred_D1 = np.clip(gbm_D1.predict(X_va_tab), 0, None) ** 2  # D1 全局 sqrt

# 对每个 worst county 训一个 sqrt 专属模型
specialists = {}
for cty in WORST_COUNTIES:
    tr_mask = (tr_loc == cty)
    va_mask = (va_loc == cty)
    n_tr, n_va = tr_mask.sum(), va_mask.sum()
    if n_tr < 100 or n_va < 10:
        print(f'  {cty}: skip (train={n_tr}, val={n_va})')
        continue
    # 小数据集 → 降低 regularization
    LOCAL = dict(max_iter=2000, learning_rate=0.03, max_depth=None,
                 max_leaf_nodes=31, min_samples_leaf=10, l2_regularization=1.0,
                 early_stopping=True, validation_fraction=0.15,
                 n_iter_no_change=30, random_state=RANDOM_SEED)
    t0 = time.time()
    g = HistGradientBoostingRegressor(loss='squared_error', **LOCAL)
    g.fit(X_tr_tab[tr_mask], np.sqrt(y_tr_raw[tr_mask]))
    p = np.clip(g.predict(X_va_tab[va_mask]), 0, None) ** 2
    specialists[cty] = (g, p, va_mask)
    # 局部 RMSE (所有 val 小时)
    local_truth = val_sorted[val_mask := va_mask]['out'].values
    local_rmse = np.sqrt(np.mean((p - local_truth)**2))
    print(f'  {cty}: train={n_tr} val={n_va} n_iter={g.n_iter_} fit={time.time()-t0:.0f}s  local_RMSE={local_rmse:.1f}')

# F2a: 基于 F1 的替换版
pred_F2a = base_pred_1d.copy()
for cty, (_, p, mask) in specialists.items():
    pred_F2a[mask] = p
Y_F2a, r_F2a = eval_pred(pred_F2a, 'F2a_F1_plus_spec')
results_78.update(r_F2a); preds_78['F2a_F1_plus_spec'] = (None, Y_F2a, pred_F2a)

# F2b: 基于 D1 sqrt 全局的替换版
pred_F2b = base_pred_D1.copy()
for cty, (_, p, mask) in specialists.items():
    pred_F2b[mask] = p
Y_F2b, r_F2b = eval_pred(pred_F2b, 'F2b_D1_plus_spec')
results_78.update(r_F2b); preds_78['F2b_D1_plus_spec'] = (None, Y_F2b, pred_F2b)

# ============================================================================
# F3: storm-specialist, 用 out>100 样本训 sqrt, 用分类器 gate
# ============================================================================
print('\n[F3] storm-hour specialist + classifier gate')
STORM_THRESH = 100
y_binary = (y_tr_raw > STORM_THRESH).astype(np.int32)
print(f'  风暴样本 (out>{STORM_THRESH}): {y_binary.sum()}/{len(y_binary)} ({y_binary.mean()*100:.1f}%)')

# 分类器: 预测是否风暴小时
t0 = time.time()
clf = HistGradientBoostingClassifier(
    max_iter=500, learning_rate=0.05, max_leaf_nodes=31,
    min_samples_leaf=20, l2_regularization=1.0,
    early_stopping=True, validation_fraction=0.15,
    n_iter_no_change=20, random_state=RANDOM_SEED)
clf.fit(X_tr_tab, y_binary)
p_storm = clf.predict_proba(X_va_tab)[:, 1]
print(f'  clf fit={time.time()-t0:.0f}s n_iter={clf.n_iter_}  p_storm[mean={p_storm.mean():.3f} max={p_storm.max():.3f}]')

# storm-specialist 回归器: 仅用 out>100 样本
t0 = time.time()
storm_mask = y_binary.astype(bool)
gbm_storm = HistGradientBoostingRegressor(loss='squared_error',
    max_iter=2000, learning_rate=0.03, max_depth=None,
    max_leaf_nodes=31, min_samples_leaf=10, l2_regularization=1.0,
    early_stopping=True, validation_fraction=0.15,
    n_iter_no_change=30, random_state=RANDOM_SEED)
gbm_storm.fit(X_tr_tab[storm_mask], np.sqrt(y_tr_raw[storm_mask]))
pred_storm_sqrt = np.clip(gbm_storm.predict(X_va_tab), 0, None) ** 2
print(f'  storm reg fit={time.time()-t0:.0f}s n_iter={gbm_storm.n_iter_}')

# F3a: hard gate (p_storm > 0.5 → storm, else D1)
pred_F3a = np.where(p_storm > 0.5, pred_storm_sqrt, base_pred_D1)
Y_F3a, r_F3a = eval_pred(pred_F3a, 'F3a_hard_gate')
results_78.update(r_F3a); preds_78['F3a_hard_gate'] = (None, Y_F3a, pred_F3a)

# F3b: soft blend (prob-weighted)
pred_F3b = p_storm * pred_storm_sqrt + (1 - p_storm) * base_pred_D1
Y_F3b, r_F3b = eval_pred(pred_F3b, 'F3b_soft_blend')
results_78.update(r_F3b); preds_78['F3b_soft_blend'] = (None, Y_F3b, pred_F3b)

# ============================================================================
# 汇总
# ============================================================================
print()
print('='*82)
print(f'{"variant":22s} {"24h":>8s} {"48h":>8s} {"full":>8s} {"Δ vs E4(88.14)":>16s}')
print('-'*82)
ref = 88.14
print(f'{"§7.7 E4 (ref)":22s} {ref:8.2f} {"--":>8s} {"--":>8s} {"(ref)":>16s}')
ordered = ['F1_onehot_sqrt','F2a_F1_plus_spec','F2b_D1_plus_spec','F3a_hard_gate','F3b_soft_blend']
for tag in ordered:
    k = f'{tag}_24h'
    if k not in results_78: continue
    r24 = results_78[k]
    r48 = results_78.get(f'{tag}_48h', float('nan'))
    rfu = results_78.get(f'{tag}_full', float('nan'))
    print(f'{tag:22s} {r24:8.2f} {r48:8.2f} {rfu:8.2f} {r24-ref:+16.2f}')
print('='*82)

# 选 global best
best_tag = min((t for t in ordered if f'{t}_24h' in results_78), key=lambda t: results_78[f'{t}_24h'])
best_24 = results_78[f'{best_tag}_24h']
print(f'\n最优 F-variant: {best_tag}  24h={best_24:.2f}')

# 保存 global best
if best_24 < all_results.get('HistGBM_24h', 88.14):
    print(f'  → 新 global best ({all_results.get("HistGBM_24h", 88.14):.2f} → {best_24:.2f}), 覆盖 results/')
    _, Y_best, _ = preds_78[best_tag]
    np.save('results/HistGBM_val_pred.npy', Y_best)
    all_results['HistGBM_24h']  = best_24
    all_results['HistGBM_48h']  = results_78[f'{best_tag}_48h']
    all_results['HistGBM_full'] = results_78[f'{best_tag}_full']
else:
    print(f'  → 没超过现有 global best, 保持')

print(f'\nPer-county ({best_tag}):')
_ = evaluate_horizons(Y_eval_true, preds_78[best_tag][1], locations, best_tag)

print()
print('='*82)
if best_24 < 85:
    print(f'[OK] {best_tag} = {best_24:.2f} < 85 目标达成, 可以锁定 Part I')
elif best_24 < ref:
    print(f'[~] 省 {ref-best_24:.2f} 点, 距 85 还差 {best_24-85:.2f}')
    print(f'    下一步: 考虑改 §1.4 (drop PCA, add raw weather lag/delta), 或接受')
else:
    print(f'[X] 所有 F-variants 都没超过 88.14, 接受 E4 为 Part I 最终')
print('='*82)

[F1] county one-hot + sqrt target
  X_tr_F1 shape: (141148, 257)  (原 174 + 83 one-hot)
  fit=42s n_iter=220
  24h=  94.25  48h=  81.51  full= 106.34  pred_max=5923

[F2] per-county specialists for 4 worst counties
  26125: train=1728 val=409 n_iter=230 fit=4s  local_RMSE=735.6
  26163: train=1728 val=409 n_iter=234 fit=4s  local_RMSE=855.6
  26161: train=1728 val=409 n_iter=537 fit=7s  local_RMSE=389.0
  26051: train=1728 val=409 n_iter=139 fit=3s  local_RMSE=193.9
  24h=  99.31  48h=  84.93  full= 106.17  pred_max=5923
  24h=  99.09  48h=  84.72  full= 106.28  pred_max=6105

[F3] storm-hour specialist + classifier gate
  风暴样本 (out>100): 6933/141148 (4.9%)
  clf fit=23s n_iter=156  p_storm[mean=0.043 max=0.991]
  storm reg fit=6s n_iter=230
  24h=  99.01  48h=  85.87  full= 103.06  pred_max=9555
  24h=  99.81  48h=  85.39  full= 101.34  pred_max=9337

variant                     24h      48h     full   Δ vs E4(88.14)
---------------------------------------------------------------------

In [ ]:
# 错误分解: 看 E4 best 的误差结构
import pandas as pd
Y_pred = np.load('results/HistGBM_val_pred.npy')  # (T_val, L_val)
err = Y_eval_true - Y_pred  # residual, 正=低估, 负=高估
abs_err = np.abs(err)

# 1. top 20 worst hours (county, time)
flat_idx = np.argsort(abs_err.flatten())[-20:][::-1]
T, L = abs_err.shape
print('Top 20 worst predictions:')
print(f'{"rank":>4s} {"t":>4s} {"loc":>6s} {"truth":>8s} {"pred":>8s} {"err":>8s}')
for rank, fi in enumerate(flat_idx):
    t, l = fi // L, fi % L
    loc_fips = locations[l]
    print(f'{rank+1:4d} {t:4d} {loc_fips:>6s} {Y_eval_true[t,l]:8.0f} {Y_pred[t,l]:8.0f} {err[t,l]:+8.0f}')

# 2. 误差按 hour 分布 (哪些时刻坏?)
hourly_rmse = np.sqrt((err**2).mean(axis=1))
worst_hours = np.argsort(hourly_rmse)[-10:][::-1]
print(f'\n最差 10 个 val hour:')
for t in worst_hours:
    print(f'  t={t:3d}  hourly_RMSE={hourly_rmse[t]:7.2f}  max_truth={Y_eval_true[t].max():.0f}  max_pred={Y_pred[t].max():.0f}')

# 3. 这些 hour 是不是少数几个风暴事件?
# 看 worst hours 在时间轴上的聚集
print(f'\nworst 10 hours 排序: {sorted(worst_hours.tolist())}')
# 如果差值 <= 3 → 属于同一风暴; >24 → 不同事件

Top 20 worst predictions:
rank    t    loc    truth     pred      err
   1   80  26163    11891      263   +11628
   2   69  26145     8704      275    +8429
   3  213  26145     8155        2    +8153
   4   81  26163    11903     3837    +8066
   5   88  26125     8088      120    +7968
   6  407  26161     7199      601    +6598
   7  405  26081     6051      220    +5831
   8  214  26145     8329     2563    +5766
   9   26  26125     4926       60    +4866
  10  406  26081     6841     2384    +4457
  11  287  26163     5097      792    +4305
  12    9  26163     4333      159    +4174
  13   52  26023     4146        1    +4145
  14   19  26125     4739      692    +4047
  15   62  26099     4511      690    +3821
  16  215  26145     7153     3420    +3733
  17  287  26125     6318     2587    +3731
  18   83  26163     6689     2960    +3729
  19  173  26125     6088     2388    +3700
  20  278  26091     3657        0    +3657

最差 10 个 val hour:
  t= 80  hourly_RMSE=1287.35  m

In [ ]:
import os, glob
# 看 Colab 当前目录和 data 子目录里有什么
print('cwd:', os.getcwd())
print('\n当前目录:')
for f in sorted(os.listdir('.')): print(' ', f)
print('\ndata/ 目录:')
if os.path.exists('data'):
    for f in sorted(os.listdir('data')): print(' ', f)
else:
    print('  不存在')
print('\n搜索所有 .nc 文件:')
for p in glob.glob('**/*.nc', recursive=True): print(' ', p)

cwd: /content

当前目录:
  .config
  drive
  results
  sample_data

data/ 目录:
  不存在

搜索所有 .nc 文件:
  drive/MyDrive/test_24h_demo.nc
  drive/MyDrive/test_48h_demo.nc
  drive/MyDrive/MLPS_Final_Project/data/train.nc


In [ ]:
import xarray as xr
for name, path in [('test_24h', 'drive/MyDrive/test_24h_demo.nc'),
                   ('test_48h', 'drive/MyDrive/test_48h_demo.nc')]:
    print(f'=== {name} ===')
    d = xr.open_dataset(path)
    print(f'  data_vars: {list(d.data_vars)}')
    for v in d.data_vars:
        print(f'    {v}: shape={d[v].shape}  dims={d[v].dims}')
    print(f'  coords: {list(d.coords)}')
    if 'timestamp' in d.coords:
        ts = d.coords['timestamp'].values
        print(f'  timestamps: n={len(ts)}  range=[{ts[0]} .. {ts[-1]}]')
    if 'feature' in d.coords:
        print(f'  feature coord: n={len(d.coords["feature"])}')
    print()

=== test_24h ===
  data_vars: ['tracked', 'out']
    tracked: shape=(83, 24)  dims=('location', 'timestamp')
    out: shape=(83, 24)  dims=('location', 'timestamp')
  coords: ['location', 'state', 'timestamp', 'feature']
  timestamps: n=24  range=[2023-06-30T01:00:00.000000000 .. 2023-07-01T00:00:00.000000000]
  feature coord: n=109

=== test_48h ===
  data_vars: ['tracked', 'out']
    tracked: shape=(83, 48)  dims=('location', 'timestamp')
    out: shape=(83, 48)  dims=('location', 'timestamp')
  coords: ['location', 'state', 'timestamp', 'feature']
  timestamps: n=48  range=[2023-06-30T01:00:00.000000000 .. 2023-07-02T00:00:00.000000000]
  feature coord: n=109



In [ ]:
import numpy as np
print('train.nc timestamps:')
print(f'  range: {ds.timestamp.values[0]} .. {ds.timestamp.values[-1]}')
print(f'  count: {len(ds.timestamp)}')

test_ts_24 = np.datetime64('2023-06-30T01:00:00')
test_ts_48 = np.datetime64('2023-07-02T00:00:00')

# 检查 test 24h/48h 是否都在 train 的时间范围内
in_range_24 = (ds.timestamp.values >= test_ts_24).sum()
in_range_48 = (ds.timestamp.values <= test_ts_48).sum()
print(f'\ntrain 里 ≥ 2023-06-30 01:00 的小时数: {in_range_24}')
print(f'train 里 ≤ 2023-07-02 00:00 的小时数: {in_range_48}')

if ds.timestamp.values[-1] >= test_ts_48:
    print('✓ train.nc 覆盖整个 test 48h 窗口 → forward weather 合法, 直接用 ds.weather')
elif ds.timestamp.values[-1] >= test_ts_24:
    print('~ 只覆盖 24h, 不覆盖 48h')
else:
    print('✗ train.nc 在 test 窗口前就结束了, forward weather 必须另想办法')

train.nc timestamps:
  range: 2023-04-01T00:00:00.000000000 .. 2023-06-30T00:00:00.000000000
  count: 2161

train 里 ≥ 2023-06-30 01:00 的小时数: 0
train 里 ≤ 2023-07-02 00:00 的小时数: 2161
✗ train.nc 在 test 窗口前就结束了, forward weather 必须另想办法


In [ ]:
# 诊断: autoregressive val 推理 — 真实 test 场景下 HistGBM 性能
import numpy as np, joblib
from copy import deepcopy

# 加载 best model (E4 ensemble 实际是两个模型的加权平均, 我们用 D1 sqrt 近似)
# D1 gbm 在 preds_76 里
gbm_ar = preds_76['D1_sqrt'][0]

# 模拟: 假设 train 末端已知, 从 val 第 1 小时开始 AR 推理
# val_sorted 已按 (timestamp, location) 排序, 每 83 行是一个时刻
L = 83  # counties
T_val_ar = T_val  # 409
print(f'AR 推理: {T_val_ar} 小时 × {L} 县 = {T_val_ar*L} 预测')

# 我们需要每一步更新 out_lag_1h. 找到 lag 列的 idx
lag_col_names = ['out_lag_1h', 'rate_lag_1h']  # 可能有更多, 先从最常见的开始
lag_idx = {c: feature_cols.index(c) for c in lag_col_names if c in feature_cols}
print(f'要更新的 lag 列 idx: {lag_idx}')
if not lag_idx:
    print('✗ feature_cols 里没找到 out_lag_1h, 改用其他诊断')

# 逐小时推理
X_ar = X_va_tab.copy()  # 要就地修改 lag
pred_ar_1d = np.zeros(len(X_ar), dtype=np.float32)
# 每 83 行一个时刻
truth_by_time = {}
for t in range(T_val_ar):
    row_start = t * L
    row_end = (t+1) * L
    if row_end > len(X_ar): break
    # 用当前 X 预测这一小时所有县
    pred_sqrt = gbm_ar.predict(X_ar[row_start:row_end])
    pred_this_hour = np.clip(pred_sqrt, 0, None) ** 2
    pred_ar_1d[row_start:row_end] = pred_this_hour
    # 更新下一小时的 lag_1h (如果下一小时存在)
    if t+1 < T_val_ar and 'out_lag_1h' in lag_idx:
        # lag_1h 在 scaler 空间, 需要用 train 的 scaler 参数逆变换
        # 简化: 直接替换原始值, 重新 scale
        # 这里先做近似: 不重新 scale, 只是标记概念
        pass  # TODO 完整实现需要记录 scaler 的 mean/std 对 lag 列

Y_pred_ar = assemble_3d(pred_ar_1d)
print(f'\nAR pred_ar range: [{Y_pred_ar.min():.1f}, {Y_pred_ar.max():.1f}]')
res_ar = rmse_horizons(Y_eval_true, Y_pred_ar, 'AR_D1')

print()
print(f'对比:')
print(f'  Val RMSE (true-lag, 当前方式): HistGBM_24h = {all_results.get("HistGBM_24h", "N/A")}')
print(f'  Val RMSE (autoregressive, 模拟 test): AR_D1_24h = {res_ar["AR_D1_24h"]:.2f}')
print(f'  差距 = autoregressive 损失')

AR 推理: 409 小时 × 83 县 = 33947 预测
要更新的 lag 列 idx: {'out_lag_1h': 0, 'rate_lag_1h': 5}

AR pred_ar range: [0.0, 6105.0]

对比:
  Val RMSE (true-lag, 当前方式): HistGBM_24h = 93.05572509765625
  Val RMSE (autoregressive, 模拟 test): AR_D1_24h = 93.45
  差距 = autoregressive 损失


In [ ]:
import numpy as np, pandas as pd

gbm_ar = preds_76['D1_sqrt'][0]
L = L_val
tv = T_val

lag_col_idx = {c: feature_cols.index(c) for c in ['out_lag_1h', 'rate_lag_1h']}
i_out_lag  = lag_col_idx['out_lag_1h']
i_rate_lag = lag_col_idx['rate_lag_1h']
sc_mean  = scaler.mean_
sc_scale = scaler.scale_

val_tracked = val_sorted['tracked'].values.astype(np.float32)
val_tracked = np.where(val_tracked <= 0, 1.0, val_tracked)

# (t, l) -> row in val_sorted
val_ts_arr = pd.to_datetime(val_sorted['timestamp']).values
val_loc_arr = val_sorted['location'].astype(str).values
ts_unique = sorted(set(val_ts_arr.tolist()))
ts2i = {t: i for i, t in enumerate(ts_unique)}
locs_str = [str(l) for l in locations]
loc2i = {l: i for i, l in enumerate(locs_str)}

rows_by_tl = np.full((tv, L), -1, dtype=np.int64)
for r in range(len(val_sorted)):
    ti = ts2i.get(val_ts_arr[r], -1)
    li = loc2i.get(val_loc_arr[r], -1)
    if ti >= 0 and li >= 0 and ti < tv and li < L:
        rows_by_tl[ti, li] = r
print(f'valid (t,l) rows: {(rows_by_tl>=0).sum()} / {tv*L}')

def ar_predict(ar_horizon):
    """AR 前 ar_horizon 小时累积 lag 误差, 之后用 true lag."""
    X = X_va_tab.copy()
    pred = np.zeros(len(X), dtype=np.float32)
    for t in range(tv):
        rows_t = rows_by_tl[t]
        valid_mask = rows_t >= 0
        if not valid_mask.any(): continue
        rs = rows_t[valid_mask]                 # row indices in X
        locs_t = np.where(valid_mask)[0]        # location idx at this timestep
        pred_sqrt = gbm_ar.predict(X[rs])
        pred_this = np.clip(pred_sqrt, 0, None) ** 2
        pred[rs] = pred_this
        if t < ar_horizon - 1 and t+1 < tv:
            # 更新下一小时的 lag_1h (对每个有 next-row 的 location)
            next_rows = rows_by_tl[t+1, locs_t]
            nmask = next_rows >= 0
            if nmask.any():
                nr = next_rows[nmask]
                pt = pred_this[nmask]
                tr = val_tracked[rs[nmask]]
                X[nr, i_out_lag]  = (pt - sc_mean[i_out_lag])  / sc_scale[i_out_lag]
                X[nr, i_rate_lag] = (pt/tr - sc_mean[i_rate_lag]) / sc_scale[i_rate_lag]
    return pred

# TrueLag 参考
pred_tl = np.clip(gbm_ar.predict(X_va_tab), 0, None) ** 2
Y_tl = assemble_3d(pred_tl)
res_tl = rmse_horizons(Y_eval_true, Y_tl, 'TrueLag')

# AR 24h block
pred_ar24 = ar_predict(24)
Y_ar24 = assemble_3d(pred_ar24)
res_ar24 = rmse_horizons(Y_eval_true, Y_ar24, 'AR_24h')

# AR 48h block
pred_ar48 = ar_predict(48)
Y_ar48 = assemble_3d(pred_ar48)
res_ar48 = rmse_horizons(Y_eval_true, Y_ar48, 'AR_48h')

# AR full
pred_arF = ar_predict(tv)
Y_arF = assemble_3d(pred_arF)
res_arF = rmse_horizons(Y_eval_true, Y_arF, 'AR_full')

print()
print('='*72)
print(f'{"scenario":28s} {"24h":>8s} {"48h":>8s} {"full":>8s}  pred_max')
print('-'*72)
for name, r, Y in [('TrueLag (val eval)', res_tl, Y_tl),
                    ('AR 24h horizon',   res_ar24, Y_ar24),
                    ('AR 48h horizon',   res_ar48, Y_ar48),
                    ('AR full (409h)',   res_arF, Y_arF)]:
    print(f'{name:28s} {r[list(r.keys())[0]]:8.2f} {r[list(r.keys())[1]]:8.2f} {r[list(r.keys())[2]]:8.2f}  {Y.max():.0f}')
print('='*72)

print()
print(f'24h 损失 (AR 24h − TrueLag) = {res_ar24["AR_24h_24h"]-res_tl["TrueLag_24h"]:+.2f}')
print(f'48h 损失 (AR 48h − TrueLag) = {res_ar48["AR_48h_48h"]-res_tl["TrueLag_48h"]:+.2f}')

valid (t,l) rows: 0 / 33947

scenario                          24h      48h     full  pred_max
------------------------------------------------------------------------
TrueLag (val eval)              93.45    81.09   106.41  6105
AR 24h horizon                 116.10   100.46   148.94  0
AR 48h horizon                 116.10   100.46   148.94  0
AR full (409h)                 116.10   100.46   148.94  0

24h 损失 (AR 24h − TrueLag) = +22.65
48h 损失 (AR 48h − TrueLag) = +19.37


In [ ]:
avail = [k for k in globals() if any(s in k for s in ['_tab','sorted','feature_cols','scaler','df_feat','df_full'])]
print(sorted(avail))
# 也看看 val_sorted / train_sorted 的列
if 'val_sorted' in globals():
    print('val_sorted cols:', list(val_sorted.columns)[:20], '...')
    print('val_sorted shape:', val_sorted.shape)
if 'feature_cols' in globals():
    print('feature_cols len:', len(feature_cols), 'first 5:', feature_cols[:5])

['X_tr_tab', 'X_va_tab', 'feature_cols', 'scaler', 'sorted_means', 'train_sorted', 'val_sorted', 'val_sorted_keys', 'y_tr_tab']
val_sorted cols: ['timestamp', 'location', 'out', 'tracked', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'out_lag_1h', 'rate_lag_1h', 'out_lag_3h', 'rate_lag_3h', 'out_lag_6h', 'rate_lag_6h', 'out_lag_12h', 'rate_lag_12h', 'out_lag_24h', 'rate_lag_24h', 'out_rmean_6h', 'out_rmax_6h'] ...
val_sorted shape: (33034, 155)
feature_cols len: 174 first 5: ['out_lag_1h', 'out_lag_3h', 'out_lag_6h', 'out_lag_12h', 'out_lag_24h']


In [ ]:
# =========================
# §7.9 AR Diagnostic (self-contained, fixed)
# =========================
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor

# ---- 0. 补齐 y_va_tab (val_sorted 行序 == X_va_tab 行序, §7.5 保证) ----
y_va_tab = val_sorted['out'].values.astype(np.float32)
assert len(y_va_tab) == X_va_tab.shape[0], f"{len(y_va_tab)} vs {X_va_tab.shape[0]}"
print(f"X_tr={X_tr_tab.shape}, X_va={X_va_tab.shape}, y_va={y_va_tab.shape}")

# ---- 1. 临时训一个 sqrt HistGBM ----
print("训练 sqrt HistGBM ...")
y_tr_sqrt = np.sqrt(np.clip(y_tr_tab, 0, None))
model_ar = HistGradientBoostingRegressor(
    loss='squared_error', max_iter=600, learning_rate=0.05,
    max_depth=8, min_samples_leaf=40, l2_regularization=0.1,
    early_stopping=True, validation_fraction=0.1,
    n_iter_no_change=30, random_state=42,
)
model_ar.fit(X_tr_tab, y_tr_sqrt)
target_mode_ar = 'sqrt'
print(f"  n_iter={model_ar.n_iter_}")

p_chk = np.clip(model_ar.predict(X_va_tab), 0, None) ** 2
rmse_chk = float(np.sqrt(np.mean((p_chk - y_va_tab) ** 2)))
print(f"Sanity val RMSE (flat): {rmse_chk:.2f}")

# ---- 2. 构造时间/县索引 (int64 ns 规范化) ----
vs = val_sorted[['timestamp','location']].copy()
vs['timestamp'] = pd.to_datetime(vs['timestamp'])
val_ts_int = vs['timestamp'].astype('int64').values
val_loc_str = vs['location'].astype(str).values

ts_unique_int = np.array(sorted(set(val_ts_int.tolist())), dtype=np.int64)
loc_unique = np.array(sorted(set(val_loc_str.tolist())))
ts2i = {int(t): i for i, t in enumerate(ts_unique_int.tolist())}
loc2i = {str(l): i for i, l in enumerate(loc_unique.tolist())}
T, L = len(ts_unique_int), len(loc_unique)
print(f"T={T}, L={L}")

rows_by_tl = np.full((T, L), -1, dtype=np.int64)
hit = 0
for r in range(len(vs)):
    ti = ts2i.get(int(val_ts_int[r]), -1)
    li = loc2i.get(val_loc_str[r], -1)
    if ti >= 0 and li >= 0:
        rows_by_tl[ti, li] = r; hit += 1
print(f"valid (t,l): {hit}/{len(vs)}")
assert hit > 0

# ---- 3. 找 lag 列索引 ----
col_names = list(feature_cols)
idx_out_lag1 = col_names.index('out_lag_1h') if 'out_lag_1h' in col_names else -1
idx_rate_lag1 = col_names.index('rate_lag_1h') if 'rate_lag_1h' in col_names else -1
print(f"idx_out_lag1={idx_out_lag1}, idx_rate_lag1={idx_rate_lag1}")

sc_mean = scaler.mean_.astype(np.float64)
sc_scale = scaler.scale_.astype(np.float64)
def fwd_scaled(x_raw, idx): return (x_raw - sc_mean[idx]) / sc_scale[idx]
def inv_target(pt): return np.clip(pt, 0, None) ** 2  # sqrt

tracked_val = val_sorted['tracked'].values.astype(np.float64) if 'tracked' in val_sorted.columns else None

# ---- 4. AR 循环 ----
X_ar = X_va_tab.copy().astype(np.float64)
y_true_2d = np.full((T, L), np.nan, dtype=np.float64)
y_pred_ar_2d = np.zeros((T, L), dtype=np.float64)

for r in range(len(vs)):
    ti = ts2i.get(int(val_ts_int[r]), -1); li = loc2i.get(val_loc_str[r], -1)
    if ti >= 0 and li >= 0:
        y_true_2d[ti, li] = y_va_tab[r]

for ti in range(T):
    rows_t = rows_by_tl[ti]
    valid = rows_t >= 0
    rs = rows_t[valid]
    if len(rs) == 0: continue
    p_count = inv_target(model_ar.predict(X_ar[rs]))
    li_idx = np.where(valid)[0]
    y_pred_ar_2d[ti, li_idx] = p_count

    if ti + 1 < T:
        rows_next = rows_by_tl[ti+1]
        for k, li in enumerate(li_idx):
            r_next = rows_next[li]
            if r_next < 0: continue
            if idx_out_lag1 >= 0:
                X_ar[r_next, idx_out_lag1] = fwd_scaled(p_count[k], idx_out_lag1)
            if idx_rate_lag1 >= 0 and tracked_val is not None:
                tr = max(tracked_val[r_next], 1.0)
                X_ar[r_next, idx_rate_lag1] = fwd_scaled(p_count[k]/tr, idx_rate_lag1)

# ---- 5. TrueLag oracle ----
p_true_count = inv_target(model_ar.predict(X_va_tab))
y_pred_true_2d = np.zeros((T, L), dtype=np.float64)
for r in range(len(vs)):
    ti = ts2i.get(int(val_ts_int[r]), -1); li = loc2i.get(val_loc_str[r], -1)
    if ti >= 0 and li >= 0:
        y_pred_true_2d[ti, li] = max(p_true_count[r], 0.0)

# ---- 6. 报告 ----
def rmse_win(yp, t0, t1):
    yt = y_true_2d[t0:t1]; pp = yp[t0:t1]
    m = ~np.isnan(yt)
    return float(np.sqrt(np.mean((yt[m]-pp[m])**2))) if m.any() else float('nan')
def rmse_storm(yp):
    m = (~np.isnan(y_true_2d)) & (y_true_2d > 50)
    return float(np.sqrt(np.mean((y_true_2d[m]-yp[m])**2))) if m.any() else float('nan')

print("\n===== AR 诊断 =====")
print(f"{'':<24}{'RMSE':>10}{'Storm>50':>12}{'峰值':>10}")
for name, yp in [("TrueLag oracle", y_pred_true_2d), ("AR full", y_pred_ar_2d)]:
    print(f"{name:<24}{rmse_win(yp,0,T):>10.2f}{rmse_storm(yp):>12.2f}{yp.max():>10.1f}")
r_true = rmse_win(y_pred_true_2d, 0, T)
r24 = rmse_win(y_pred_ar_2d, 0, min(24,T))
r48 = rmse_win(y_pred_ar_2d, 0, min(48,T))
rfull = rmse_win(y_pred_ar_2d, 0, T)
print(f"AR 24h: {r24:.2f}  退化 {r24-r_true:+.2f}")
print(f"AR 48h: {r48:.2f}  退化 {r48-r_true:+.2f}")
print(f"AR full: {rfull:.2f}  退化 {rfull-r_true:+.2f}")
print("\n判据: <10 继续刷val / 10-30 补残差 / >30 换multi-step")

X_tr=(141148, 174), X_va=(33034, 174), y_va=(33034,)
训练 sqrt HistGBM ...
  n_iter=120
Sanity val RMSE (flat): 188.75
T=409, L=83
valid (t,l): 33034/33034
idx_out_lag1=0, idx_rate_lag1=5

===== AR 诊断 =====
                              RMSE    Storm>50        峰值
TrueLag oracle              188.75      725.31    6018.1
AR full                     257.84      996.30    5365.9
AR 24h: 289.81  退化 +101.06
AR 48h: 246.02  退化 +57.28
AR full: 257.84  退化 +69.09

判据: <10 继续刷val / 10-30 补残差 / >30 换multi-step


In [ ]:
print("y_tr_tab:", y_tr_tab.min(), y_tr_tab.mean(), y_tr_tab.max())
print("val_sorted['out']:", val_sorted['out'].min(), val_sorted['out'].mean(), val_sorted['out'].max())
print("train_sorted['out']:", train_sorted['out'].min(), train_sorted['out'].mean(), train_sorted['out'].max())

y_tr_tab: 0.0 46.6576 23346.0
val_sorted['out']: 0.0 32.912 11903.0
train_sorted['out']: 0.0 46.6576 23346.0


In [ ]:
if SKIP_LSTM_TRAIN:
    print('[SKIPPED] §7.4 leaderboard viz (SKIP_LSTM_TRAIN=True)')
else:
    # §7.4 排行榜可视化
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    for ax, h in zip(axes, ['24h', '48h']):
        hr = sorted([(k,v) for k,v in all_results.items() if h in k], key=lambda x: x[1])
        if not hr: continue
        names, rmses = zip(*hr)
        colors = ['#e74c3c' if MODEL_NAME in n else '#2196F3' if 'SARIMAX' in n
                  else '#4CAF50' if 'GBM' in n else '#FF9800' if 'Seq2Seq' in n else '#bdbdbd' for n in names]
        bars = ax.barh(range(len(names)), rmses, color=colors, alpha=.85)
        ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=10)
        ax.set_xlabel('县均 RMSE'); ax.set_title(f'{h}', fontsize=13, fontweight='bold')
        ax.invert_yaxis()
        for i, (b, v) in enumerate(zip(bars, rmses)):
            ax.text(v + max(rmses)*.01, i, f'{v:.2f}', va='center', fontsize=9)
    plt.suptitle(f'排行榜 — {MODEL_NAME}', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()
    if WANDB_ENABLED: wandb.log({'charts/leaderboard': wandb.Image(fig)})

---
## §P2: Phase 2 — 加入 GNN 空间模块

上面的 Phase 1 结果是纯 LSTM 的。现在开启 GNN，重新训练，看空间信息能提升多少。

In [ ]:
if SKIP_PART2_GNN:
    print('[SKIPPED] §P2.1 GNN DataLoader (SKIP_PART2_GNN=True)')
else:
    # §P2.1: 重新构建 GNN 版本的 DataLoader + 模型

    print("=" * 60)
    print("Phase 2: Deep LSTM + GNN")
    print("=" * 60)

    # 切换到 all_counties 模式
    GNN_ENABLED_P2 = True
    DS_MODE_P2 = 'all_counties'

    train_ds_gnn = OutageDataset(X_train_3d, Y_train_3d, SEQ_LEN, HORIZON, mode=DS_MODE_P2)
    val_ds_gnn   = OutageDataset(X_val_3d,   Y_val_3d,   SEQ_LEN, HORIZON, mode=DS_MODE_P2)
    train_loader_gnn = DataLoader(train_ds_gnn, batch_size=min(BATCH_SIZE, 16), shuffle=True, num_workers=0)
    val_loader_gnn   = DataLoader(val_ds_gnn,   batch_size=min(BATCH_SIZE, 32), shuffle=False, num_workers=0)

    xb_g, yb_g = next(iter(train_loader_gnn))
    print(f"GNN DataLoader: X={xb_g.shape} Y={yb_g.shape}")
    print(f"  X = (batch, seq_len={SEQ_LEN}, counties={L}, features={len(feature_cols)})")
    print(f"  Y = (batch, horizon={HORIZON}, counties={L})")

    # 构建邻接矩阵
    adj_matrix_gnn = build_adj(locations, GNN_K).to(DEVICE)
    print(f"\n邻接矩阵: {adj_matrix_gnn.shape}")

    # 用 Phase 1 训好的模型作为 backbone (迁移学习)
    # 根据 MODEL_TYPE 创建对应的 backbone
    if MODEL_TYPE == 'simple':
        backbone_gnn = SimpleSeq2Seq(INPUT_DIM, HIDDEN_DIM, NUM_LAYERS, HORIZON).to(DEVICE)
    elif MODEL_TYPE == 'lstm':
        backbone_gnn = DeepLSTM(INPUT_DIM, HIDDEN_DIM, NUM_LAYERS, HORIZON,
                                DROPOUT, USE_GRU, BIDIRECTIONAL).to(DEVICE)
    elif MODEL_TYPE == 'seq2seq':
        backbone_gnn = Seq2SeqAttention(INPUT_DIM, HIDDEN_DIM, NUM_LAYERS, HORIZON, DROPOUT).to(DEVICE)
    elif MODEL_TYPE == 'transformer':
        backbone_gnn = TransformerForecaster(INPUT_DIM, HIDDEN_DIM, N_HEADS, NUM_LAYERS, HORIZON,
                                             DIM_FEEDFORWARD, DROPOUT, causal=USE_CAUSAL_MASK).to(DEVICE)
    else:
        raise ValueError(f'Unknown MODEL_TYPE: {MODEL_TYPE}')

    # 加载 Phase 1 的权重到 backbone
    phase1_ckpt = torch.load(os.path.join(RESULTS_DIR, f'{MODEL_NAME}_best.pt'),
                              map_location=DEVICE, weights_only=True)
    backbone_gnn.load_state_dict(phase1_ckpt)
    print("✓ 加载 Phase 1 LSTM 权重 (迁移学习)")

    # 构建 GNN 模型
    model_gnn = SpatialWrapper(
        backbone_gnn, HIDDEN_DIM, HORIZON, L,
        gnn_hidden=GNN_HIDDEN, fusion=SPATIAL_FUSION, dropout=DROPOUT
    ).to(DEVICE)

    n_params_gnn = sum(p.numel() for p in model_gnn.parameters() if p.requires_grad)
    print(f"\nLSTM+GNN 参数量: {n_params_gnn:,} (vs Phase 1: {n_params:,}, +{n_params_gnn-n_params:,})")

In [ ]:
if SKIP_PART2_GNN:
    print('[SKIPPED] §P2.2 LSTM+GNN training (SKIP_PART2_GNN=True)')
else:
    # §P2.2: 训练 LSTM+GNN

    GNN_MODEL_NAME = "DeepLSTM_GNN"
    GNN_EPOCHS = 60
    GNN_LR = 2e-4  # GNN 部分用更小的学习率 (backbone 已预训练)

    optimizer_gnn = optim.AdamW([
        {'params': model_gnn.backbone.parameters(), 'lr': GNN_LR * 0.1},  # backbone 低学习率 (微调)
        {'params': model_gnn.gcn1.parameters(), 'lr': GNN_LR},
        {'params': model_gnn.gcn2.parameters(), 'lr': GNN_LR},
        {'params': model_gnn.head.parameters(), 'lr': GNN_LR},
    ] + ([{'params': model_gnn.gate_fc.parameters(), 'lr': GNN_LR},
          {'params': model_gnn.gnn_proj.parameters(), 'lr': GNN_LR}]
         if hasattr(model_gnn, 'gate_fc') else []),
        weight_decay=WEIGHT_DECAY)

    scheduler_gnn = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_gnn, mode='min', factor=0.5, patience=3, min_lr=1e-6)

    best_gnn_rmse, best_gnn_epoch, patience_gnn = float('inf'), 0, 0
    history_gnn = {'train_loss':[], 'val_loss':[], 'val_rmse':[], 'lr':[]}
    t_start_gnn = time.time()

    print(f"Training LSTM+GNN | {GNN_EPOCHS} epochs\n")

    for epoch in range(1, GNN_EPOCHS + 1):
        model_gnn.train()
        losses = []
        for xb, yb in train_loader_gnn:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)     # xb: (B,T,L,F), yb: (B,H,L)

            # ---- Mixup ----
            if MIXUP_ALPHA > 0:
                lam = np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)
                lam = max(lam, 1 - lam)
                perm = torch.randperm(xb.size(0), device=xb.device)
                xb = lam * xb + (1 - lam) * xb[perm]
                yb = lam * yb + (1 - lam) * yb[perm]

            optimizer_gnn.zero_grad()
            pred = model_gnn(xb, adj_matrix_gnn)        # (B, L, H)
            pred = pred.permute(0, 2, 1)                 # (B, H, L) — 对齐 yb
            loss = criterion(pred, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model_gnn.parameters(), GRADIENT_CLIP)
            optimizer_gnn.step()
            losses.append(loss.item())
        train_loss = np.mean(losses)

        model_gnn.eval()
        vl, vp, vt_list = [], [], []
        with torch.no_grad():
            for xb, yb in val_loader_gnn:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                pred = model_gnn(xb, adj_matrix_gnn).permute(0, 2, 1)
                vl.append(criterion(pred, yb).item())
                vp.append(pred.cpu().numpy()); vt_list.append(yb.cpu().numpy())
        val_loss = np.mean(vl)
        all_p, all_t = np.concatenate(vp), np.concatenate(vt_list)
        val_rmse = np.sqrt(np.mean((all_p - all_t)**2))

        cur_lr = optimizer_gnn.param_groups[0]['lr']
        scheduler_gnn.step(val_rmse)

        history_gnn['train_loss'].append(train_loss)
        history_gnn['val_loss'].append(val_loss)
        history_gnn['val_rmse'].append(val_rmse)
        history_gnn['lr'].append(cur_lr)

        if WANDB_ENABLED:
            wandb.log({'epoch_gnn': epoch, 'gnn/train_loss': train_loss,
                       'gnn/val_loss': val_loss, 'gnn/val_rmse': val_rmse})

        if val_rmse < best_gnn_rmse:
            best_gnn_rmse, best_gnn_epoch, patience_gnn = val_rmse, epoch, 0
            torch.save(model_gnn.state_dict(), os.path.join(RESULTS_DIR, f'{GNN_MODEL_NAME}_best.pt'))
        else:
            patience_gnn += 1

        if epoch % 5 == 0 or epoch == 1 or patience_gnn == 0:
            m = '★' if patience_gnn == 0 else ' '
            print(f'  E{epoch:3d}/{GNN_EPOCHS} | Loss:{train_loss:.4f} ValRMSE:{val_rmse:.4f} '
                  f'Best:{best_gnn_rmse:.4f}@{best_gnn_epoch} {m}')

        if patience_gnn >= EARLY_STOPPING_PATIENCE:
            print(f'\n  ⏹ Early stop @ epoch {epoch}')
            break

    total_gnn_time = time.time() - t_start_gnn
    print(f'\n✓ GNN训练完成 {total_gnn_time:.0f}s | Best: {best_gnn_rmse:.4f} @ epoch {best_gnn_epoch}')

In [ ]:
if SKIP_PART2_GNN:
    print('[SKIPPED] §P2.3 GNN val inference (SKIP_PART2_GNN=True)')
else:
    # §P2.3: GNN 验证集预测 + 评估

    model_gnn.load_state_dict(torch.load(os.path.join(RESULTS_DIR, f'{GNN_MODEL_NAME}_best.pt'),
                                          map_location=DEVICE, weights_only=True))
    model_gnn.eval()

    Y_pred_gnn = np.zeros_like(Y_val_3d)  # (T_val, L)
    counts_gnn = np.zeros_like(Y_val_3d)

    with torch.no_grad():
        X_full = np.concatenate([X_train_3d[-SEQ_LEN:], X_val_3d], axis=0)  # (SEQ+T_val, L, F)

        for t in range(0, X_full.shape[0] - SEQ_LEN - HORIZON + 1, HORIZON):
            xin = torch.tensor(X_full[t:t+SEQ_LEN], dtype=torch.float32).unsqueeze(0).to(DEVICE)
            pred = model_gnn(xin, adj_matrix_gnn)  # (1, L, HORIZON)
            pred = pred.squeeze(0).permute(1, 0).cpu().numpy()  # (HORIZON, L)

            vs = t
            ve = min(vs + HORIZON, Y_val_3d.shape[0])
            nf = ve - max(vs, 0)
            if vs >= 0 and nf > 0:
                Y_pred_gnn[vs:ve] += pred[:nf]
                counts_gnn[vs:ve] += 1

    counts_gnn[counts_gnn == 0] = 1
    Y_pred_gnn = Y_pred_gnn / counts_gnn

    # 根据 target mode 还原到原始空间 (与 §7.2 保持一致)
    if TARGET_MODE == 'log1p':
        Y_pred_gnn = np.expm1(np.clip(Y_pred_gnn, 0, 20))
        print('✓ GNN 预测: log1p → expm1 还原完成')
    elif TARGET_MODE == 'rate':
        Y_pred_gnn = np.clip(Y_pred_gnn, 0, 1) * TRK_val_3d
        print('✓ GNN 预测: rate × tracked → 原始停电数还原完成')

    Y_pred_gnn = np.clip(Y_pred_gnn, 0, None)

    # 保存
    np.save(os.path.join(RESULTS_DIR, f'{GNN_MODEL_NAME}_val_pred.npy'), Y_pred_gnn)

    # 评估 (原始空间, 与 Phase 1 用同一口径)
    Y_eval_true = Y_val_3d_raw
    print(f'Y_eval_true range: [{Y_eval_true.min():.1f}, {Y_eval_true.max():.1f}] (原始停电数)')
    print(f'Y_pred_gnn  range: [{Y_pred_gnn.min():.1f}, {Y_pred_gnn.max():.1f}]')
    gnn_results, gnn_county_rmses = evaluate_horizons(Y_eval_true, Y_pred_gnn, locations, GNN_MODEL_NAME)
    print()


In [ ]:
if SKIP_PART2_GNN:
    print('[SKIPPED] §P2.4 Phase1 vs Phase2 (SKIP_PART2_GNN=True)')
else:
    # §P2.4: Phase 1 vs Phase 2 对比

    print("=" * 70)
    print("Phase 1 (LSTM only) vs Phase 2 (LSTM + GNN) 对比")
    print("=" * 70)

    # 合并所有结果 (BASELINES_LIVE 在 §7.3 生成, 是当前 val 上真算的 baseline)
    baselines_dict = BASELINES_LIVE if 'BASELINES_LIVE' in globals() else {}
    all_final = {**baselines_dict, **model_results, **gnn_results}

    for h in ['24h', '48h']:
        print(f"\n--- {h} ---")
        hr = {k:v for k,v in all_final.items() if h in k}
        for name, rmse in sorted(hr.items(), key=lambda x: x[1]):
            tag = ''
            if 'GNN' in name: tag = '  ◀ LSTM+GNN'
            elif MODEL_NAME in name: tag = '  ◀ LSTM only'
            print(f"  {name:30s}: {rmse:10.4f}{tag}")

    # 逐县对比
    print(f"\n{'='*70}")
    print("逐县 RMSE 对比: LSTM vs LSTM+GNN")
    print(f"{'='*70}")

    lstm_wins, gnn_wins, ties = 0, 0, 0
    diffs = []
    for loc in locations:
        r_lstm = model_county_rmses.get(loc, float('inf'))
        r_gnn = gnn_county_rmses.get(loc, float('inf'))
        diff = r_lstm - r_gnn  # 正数 = GNN 更好
        diffs.append(diff)
        if diff > 0.1: gnn_wins += 1
        elif diff < -0.1: lstm_wins += 1
        else: ties += 1

    print(f"  GNN 更好: {gnn_wins}/{L} 县")
    print(f"  LSTM 更好: {lstm_wins}/{L} 县")
    print(f"  持平: {ties}/{L} 县")
    print(f"  RMSE 差值: mean={np.mean(diffs):.2f}, median={np.median(diffs):.2f}")

    # 可视化
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    # 散点图
    ax = axes[0]
    x_lstm = [model_county_rmses[loc] for loc in locations]
    y_gnn = [gnn_county_rmses[loc] for loc in locations]
    ax.scatter(x_lstm, y_gnn, alpha=0.6, s=40, c='#2196F3', edgecolors='white')
    maxv = max(max(x_lstm), max(y_gnn))
    ax.plot([0, maxv], [0, maxv], 'k--', alpha=0.3, label='y=x')
    ax.set_xlabel('LSTM-only RMSE'); ax.set_ylabel('LSTM+GNN RMSE')
    ax.set_title(f'各县 RMSE: LSTM vs LSTM+GNN\n(对角线下方 = GNN 更好, {gnn_wins}/{L} 县)')
    ax.legend()

    # RMSE 差值排序
    ax = axes[1]
    sorted_diffs = sorted(zip(locations, diffs), key=lambda x: x[1], reverse=True)
    colors_d = ['#4CAF50' if d > 0 else '#F44336' for _, d in sorted_diffs]
    ax.barh(range(L), [d for _, d in sorted_diffs], color=colors_d, alpha=0.7)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.set_xlabel('RMSE 差值 (LSTM - GNN, 正=GNN更好)')
    ax.set_title('各县: GNN 带来的 RMSE 变化')
    ax.set_yticks([])

    # 训练曲线对比
    ax = axes[2]
    ax.plot(history['val_rmse'], label='LSTM only', color='#FF9800', alpha=0.8)
    ax.plot(history_gnn['val_rmse'], label='LSTM+GNN', color='#2196F3', alpha=0.8)
    ax.axhline(best_val_rmse, color='#FF9800', ls='--', alpha=0.4)
    ax.axhline(best_gnn_rmse, color='#2196F3', ls='--', alpha=0.4)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Val RMSE'); ax.set_title('训练曲线对比')
    ax.legend()

    plt.suptitle(f'Deep LSTM vs LSTM+GNN | GNN 提升 {gnn_wins}/{L} 县', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

    if WANDB_ENABLED:
        wandb.log({'charts/lstm_vs_gnn': wandb.Image(fig)})
        for k, v in gnn_results.items():
            wandb.summary[f'eval_gnn/{k}'] = v


In [ ]:
if SKIP_PART2_GNN:
    print('[SKIPPED] §P2.5 final summary (SKIP_PART2_GNN=True)')
else:
    # §P2.5: 最终总结

    print(f"\n{'='*70}")
    print("实验总结")
    print(f"{'='*70}")
    print(f"\nPhase 1: {MODEL_NAME}")
    print(f"  Best Val RMSE: {best_val_rmse:.4f} @ epoch {best_epoch}")
    print(f"  参数量: {n_params:,}")
    for k,v in sorted(model_results.items(), key=lambda x: x[1]):
        print(f"  {k}: {v:.4f}")

    print(f"\nPhase 2: {GNN_MODEL_NAME}")
    print(f"  Best Val RMSE: {best_gnn_rmse:.4f} @ epoch {best_gnn_epoch}")
    print(f"  参数量: {n_params_gnn:,}")
    for k,v in sorted(gnn_results.items(), key=lambda x: x[1]):
        print(f"  {k}: {v:.4f}")

    lstm_24 = model_results.get(f'{MODEL_NAME}_24h', float('inf'))
    gnn_24 = gnn_results.get(f'{GNN_MODEL_NAME}_24h', float('inf'))
    print(f"\nGNN 在 24h 上的提升: {lstm_24:.2f} → {gnn_24:.2f} ({(lstm_24-gnn_24)/lstm_24*100:+.1f}%)")

    print(f"\n交付物:")
    print(f"  {RESULTS_DIR}/{MODEL_NAME}_best.pt")
    print(f"  {RESULTS_DIR}/{MODEL_NAME}_val_pred.npy")
    print(f"  {RESULTS_DIR}/{GNN_MODEL_NAME}_best.pt")
    print(f"  {RESULTS_DIR}/{GNN_MODEL_NAME}_val_pred.npy")

    if WANDB_ENABLED:
        wandb.summary['phase1_best_rmse'] = best_val_rmse
        wandb.summary['phase2_best_rmse'] = best_gnn_rmse
        wandb.summary['gnn_improvement_24h'] = lstm_24 - gnn_24
        wandb.finish()
        print("\n✓ wandb finished")

---
## §8 提交文件生成

In [ ]:
if SKIP_PART2_GNN:
    print('[SKIPPED] §8 submission files (SKIP_PART2_GNN=True)')
else:
    # §8 提交文件
    GENERATE_SUBMISSION = False  # TODO: 确认模型效果后改为 True

    if GENERATE_SUBMISSION:
        # TODO: 加载 test data + 推理 + 生成 CSV
        # ds_test_24h = xr.open_dataset(os.path.join(DATA_DIR, 'test_24h_demo.nc'))
        # ds_test_48h = xr.open_dataset(os.path.join(DATA_DIR, 'test_48h_demo.nc'))
        # ...
        print('⚠ 请完成 test set 推理代码')
    else:
        print('跳过 (GENERATE_SUBMISSION=False)')

In [ ]:
if SKIP_PART2_GNN:
    print('[SKIPPED] §9 (deprecated) (SKIP_PART2_GNN=True)')
else:
    # §9 (被 §P2.5 替代)
    print("见上方 §P2.5 总结")

---
## 附录

### 调参优先级
1. **Loss** — `huber` 或 `weighted_mse` (极端值友好)
2. **LR** — {1e-4, 3e-4, 1e-3, 3e-3}，Transformer 偏小
3. **Depth** — layers × hidden_dim
4. **SEQ_LEN** — {24, 48, 72}，Attention/Transformer 受益于长序列
5. **进阶模块** — GNN, Two-Stage, Ensemble

### Phase 1 关键发现
- 停电滞后特征贡献 60%+ → 确保 `OUTAGE_LAGS` 包含 24h
- 极端县 (26125, 26163) RMSE 极高 → 尝试县级 re-weighting
- 70% 零值 → Two-Stage (先分类再回归) 可能有效
- 天气 rolling 特征比 raw 更有用 → `weather_rolling` 组

### 分工交付协议
每人完成后提供:
1. `results/{MODEL_NAME}_best.pt` — checkpoint
2. `results/{MODEL_NAME}_val_pred.npy` — 验证集预测 `(T_val, L)`
3. wandb 上的完整实验记录
4. 24h / 48h RMSE 数字 → 汇总到排行榜